In [1]:
!pip install -q transformers>=4.57.0 accelerate pillow opencv-python-headless qwen-vl-utils huggingface_hub scikit-image -U bitsandbytes>=0.46.1

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.1.1 which is incompatible.
cucim-cu12 26.2.0 requires scikit-image<0.26.0,>=0.19.0, but you have scikit-image 0.26.0 which is incompatible.


In [2]:
!pip install -q transformers --upgrade

In [ ]:
!pip install -q pillow --upgrade

Trying to process the video at once or in large chunks

In [ ]:
!pip install -q git+https://github.com/huggingface/transformers.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
# ===== Dashcam Analyzer — Clip Mode v10 (overlap, no context) =====

import torch
import cv2
import numpy as np
import gc
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText
from qwen_vl_utils import process_vision_info
from difflib import SequenceMatcher
import time

VIDEO_PATH = "/content/dashcam.mp4"
MODEL_ID   = "Qwen/Qwen3-VL-8B-Instruct"

# --------------------------------------------------
# Load model
# --------------------------------------------------

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()

# --------------------------------------------------
# CONFIG
# --------------------------------------------------

start_time      = time.time()
CLIP_DURATION   = 6.0
OVERLAP         = 2.0
NFRAMES         = 16
MAX_TOKENS      = 500
MEMORY_CHARS    = 80
FLOW_FPS        = 8
DEDUP_THRESHOLD = 0.6

# --------------------------------------------------
# VIDEO INFO
# --------------------------------------------------

cap          = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise RuntimeError(f"Could not open video: {VIDEO_PATH}")

native_fps   = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

if native_fps == 0:
    raise RuntimeError("Could not read FPS — check video file format/codec.")

duration = total_frames / native_fps
print(f"Video duration: {duration:.2f}s  |  FPS: {native_fps}  |  Frames: {total_frames}")

# --------------------------------------------------
# READ ALL FRAMES ONCE
# --------------------------------------------------

frame_interval = max(1, int(native_fps / FLOW_FPS))
all_frames     = []
frame_idx      = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break
    if frame_idx % frame_interval == 0:
        ts   = frame_idx / native_fps
        gray = cv2.cvtColor(cv2.resize(frame, (640, 360)), cv2.COLOR_BGR2GRAY)
        rgb  = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        all_frames.append((ts, gray, Image.fromarray(rgb)))
    frame_idx += 1

cap.release()
print(f"Frames loaded: {len(all_frames)}")

timestamps = np.array([f[0] for f in all_frames])

# --------------------------------------------------
# BUILD CLIPS
# --------------------------------------------------

step = CLIP_DURATION - OVERLAP
clip_starts = np.arange(0, duration - OVERLAP, step)
clips  = [(round(cs, 2), round(min(cs + CLIP_DURATION, duration), 2)) for cs in clip_starts]

print(f"Total clips: {len(clips)}")

# --------------------------------------------------
# FRAME SAMPLER
# --------------------------------------------------

def get_clip_frames(start, end, n=NFRAMES):
    mask    = (timestamps >= start) & (timestamps <= end)
    indices = np.where(mask)[0]
    if len(indices) == 0:
        return []
    sampled = np.linspace(0, len(indices) - 1, min(n, len(indices)), dtype=int)
    return [all_frames[indices[i]][2] for i in sampled]

def get_reference_frame(start):
    mask    = timestamps >= start
    indices = np.where(mask)[0]
    if len(indices) == 0:
        return None
    return all_frames[indices[0]][2]

# --------------------------------------------------
# DEDUPLICATION
# --------------------------------------------------

def deduplicate_entries(text):
    lines   = [l.strip() for l in text.splitlines() if l.strip()]
    cleaned = []
    for line in lines:
        if not cleaned:
            cleaned.append(line)
            continue
        ratio = SequenceMatcher(None, line.lower(), cleaned[-1].lower()).ratio()
        if ratio < DEDUP_THRESHOLD:
            cleaned.append(line)
    return "\n".join(cleaned)

# --------------------------------------------------
# PROMPTS
# --------------------------------------------------

SYSTEM_PROMPT = """You are an expert dashcam footage analyst.

CRITICAL: Only describe what is directly visible in the provided frames. Do NOT infer or carry forward objects not visible.

CAMERA: This footage is from a moving vehicle. Judge ALL vehicle movements relative to the road, not the camera.
Left = driver's left (oncoming traffic). Right = driver's right (shoulder/curb).

For each person: describe clothing, identify occupation, describe action and location.
For each vehicle: identify type, position on road, movement direction, lights.
Note all road infrastructure and abnormal events.
Never speculate. Only describe what is visible."""

PROMPT = """Reference frame shown above — use it as spatial anchor.

Segment {start}-{end}s. Write a mini-timeline, one entry per change:
[timestamp] one detailed sentence

Write {target} entries. Timestamps within {start}-{end}s.
Vehicles leaving road, smoke, people crossing lanes each get their own entry.
Only describe what is directly visible — do NOT infer or carry forward objects not visible in these frames."""

# --------------------------------------------------
# MODEL INFERENCE
# --------------------------------------------------

def analyze_clip(start, end):
    clip_frames = get_clip_frames(start, end)
    ref_frame   = get_reference_frame(start)
    target      = max(3, int(CLIP_DURATION))

    if not clip_frames:
        return "No frames available."

    prompt = PROMPT.format(
        start=start,
        end=end,
        target=target,
    )

    content = (
        ([{"type": "image", "image": ref_frame}] if ref_frame else [])
        + [{"type": "image", "image": f} for f in clip_frames]
        + [{"type": "text", "text": prompt}]
    )

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": content}
    ]

    text_inp        = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, _ = process_vision_info(messages)

    inputs = processor(
        text=[text_inp],
        images=image_inputs,
        padding=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_TOKENS,
            do_sample=False,
        )

    trimmed = output[:, inputs["input_ids"].shape[1]:]
    result  = processor.batch_decode(trimmed, skip_special_tokens=True)[0]

    del inputs, output
    gc.collect()
    torch.cuda.empty_cache()

    return deduplicate_entries(result.strip())

# --------------------------------------------------
# RUN PIPELINE
# --------------------------------------------------

timeline = []

for i, (start, end) in enumerate(clips):
    print(f"Clip {i+1}/{len(clips)} [{start}-{end}s]")
    text = analyze_clip(start, end)
    timeline.append((start, end, text))

# --------------------------------------------------
# PRINT TIMELINE
# --------------------------------------------------

print("\nTIMELINE\n")
for s, e, t in timeline:
    print(f"[{s}-{e}s]\n{t}\n")

print(f"\nTotal runtime: {time.time() - start_time:.1f}s")

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

Video duration: 69.53s  |  FPS: 30.0  |  Frames: 2086
Frames extracted: 694


KeyboardInterrupt: 

Dynamic peaks but changing 2 'best' frames for GAP instead of 4 like peaks

In [27]:
# ===== Dashcam Analyzer v28b — Content-Driven + Clamped Peaks =====
# Changes from v28:
#   - Peak chunks clamped to max 4 seconds (±2s around peak)
#   - Gap chunks max raised to 10 seconds
#   - Leftover space from clamped peaks becomes gap chunks

import torch
import cv2
import numpy as np
import gc
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText
from qwen_vl_utils import process_vision_info
from scipy.signal import find_peaks
import time

VIDEO_PATH = "/content/dashcam.mp4"
MODEL_ID   = "Qwen/Qwen3-VL-8B-Instruct"

# --------------------------------------------------
# Load model
# --------------------------------------------------

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

# --------------------------------------------------
# CONFIG
# --------------------------------------------------

start_time = time.time()

FLOW_FPS       = 8
MAX_TOKENS     = 120
MEMORY_CHARS   = 100
ROLLING_WINDOW = 8

# Peak detection
PEAK_MIN_DISTANCE_S = 1.5
PEAK_CHUNK_PAD_S    = 1.5
PEAK_PERCENTILE     = 75

# Chunk sizing
MAX_PEAK_CHUNK_S    = 4.0    # max peak chunk duration — clamp midpoint chunks
MAX_GAP_CHUNK_S     = 10.0   # max gap chunk duration — static sections
MIN_GAP_CHUNK_S     = 1.0    # minimum gap worth describing

# --------------------------------------------------
# VIDEO INFO
# --------------------------------------------------

cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise RuntimeError(f"Could not open video: {VIDEO_PATH}")

native_fps   = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

if native_fps == 0:
    raise RuntimeError("Could not read FPS — check video file format/codec.")

duration = total_frames / native_fps
print(f"Video duration: {duration:.2f}s  |  FPS: {native_fps}  |  Frames: {total_frames}")

# --------------------------------------------------
# FRAME EXTRACTION
# --------------------------------------------------

frame_interval = max(1, int(native_fps / FLOW_FPS))
frames         = []
frame_idx      = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break
    if frame_idx % frame_interval == 0:
        ts         = frame_idx / native_fps
        rgb        = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        gray       = cv2.cvtColor(cv2.resize(frame, (640, 360)), cv2.COLOR_BGR2GRAY)
        brightness = float(gray.mean())
        frames.append((ts, gray, brightness, rgb))
    frame_idx += 1

cap.release()
print(f"Frames extracted: {len(frames)}")

frame_timestamps = np.array([f[0] for f in frames])

# --------------------------------------------------
# SIGNAL COMPUTATION
# --------------------------------------------------

flow_variance_scores = [0.0]
flow_collapse_scores = [0.0]
prev_forward_flow    = None

for i in range(1, len(frames)):
    prev_gray = frames[i-1][1]
    curr_gray = frames[i][1]

    flow = cv2.calcOpticalFlowFarneback(
        prev_gray, curr_gray, None,
        0.5, 3, 15, 3, 5, 1.2, 0
    )

    mag, _        = cv2.cartToPolar(flow[..., 0], flow[..., 1])
    flow_variance = float(np.std(mag))
    flow_variance_scores.append(flow_variance)

    forward_flow = float(np.mean(np.abs(flow[..., 1])))
    collapse     = max(0.0, prev_forward_flow - forward_flow) if prev_forward_flow is not None else 0.0
    flow_collapse_scores.append(collapse)
    prev_forward_flow = forward_flow

flow_var_arr      = np.array(flow_variance_scores)
flow_collapse_arr = np.array(flow_collapse_scores)

brightness_arr    = np.array([f[2] for f in frames])
d1                = np.abs(np.diff(brightness_arr))
d2                = np.abs(np.diff(d1))
brightness_scores = np.concatenate([[0, 0], d2])

scene_delta_scores = []
for i in range(len(frames)):
    curr_gray    = frames[i][1].astype(np.float32)
    window_start = max(0, i - ROLLING_WINDOW)
    window_grays = np.array([frames[j][1].astype(np.float32) for j in range(window_start, i)])

    if len(window_grays) == 0:
        scene_delta_scores.append(0.0)
    else:
        rolling_mean = window_grays.mean(axis=0)
        scene_delta_scores.append(float(np.mean(np.abs(curr_gray - rolling_mean))))

scene_delta_arr = np.array(scene_delta_scores)

importance = flow_var_arr + flow_collapse_arr + brightness_scores + scene_delta_arr

# --------------------------------------------------
# FIND PEAKS
# --------------------------------------------------

min_distance_frames = max(1, int(PEAK_MIN_DISTANCE_S * FLOW_FPS))
height_threshold = float(np.percentile(importance, PEAK_PERCENTILE))

peak_indices, _ = find_peaks(
    importance,
    height=height_threshold,
    distance=min_distance_frames
)

peak_times = frame_timestamps[peak_indices]

print(f"\n--- PEAK DETECTION ---")
print(f"Importance threshold (p{PEAK_PERCENTILE}): {height_threshold:.4f}")
print(f"Min peak distance: {PEAK_MIN_DISTANCE_S}s ({min_distance_frames} frames)")
print(f"Peaks found: {len(peak_indices)}")
for pt in peak_times:
    print(f"  Peak at {pt:.2f}s")

# --------------------------------------------------
# BUILD CHUNKS — midpoint splitting + clamping
# --------------------------------------------------

# Step 1: Compute midpoint boundaries
midpoint_chunks = []

if len(peak_times) > 0:
    for i, pt in enumerate(peak_times):
        if i == 0:
            cs = max(0.0, pt - PEAK_CHUNK_PAD_S)
        else:
            cs = (peak_times[i - 1] + pt) / 2.0

        if i == len(peak_times) - 1:
            ce = min(duration, pt + PEAK_CHUNK_PAD_S)
        else:
            ce = (pt + peak_times[i + 1]) / 2.0

        midpoint_chunks.append((pt, cs, ce))  # store peak time too

# Step 2: Clamp each chunk to MAX_PEAK_CHUNK_S around its peak
peak_chunks = []
for pt, cs, ce in midpoint_chunks:
    half_max = MAX_PEAK_CHUNK_S / 2.0
    clamped_cs = max(cs, pt - half_max)
    clamped_ce = min(ce, pt + half_max)
    # Also clamp to video bounds
    clamped_cs = max(0.0, clamped_cs)
    clamped_ce = min(duration, clamped_ce)
    peak_chunks.append((round(clamped_cs, 2), round(clamped_ce, 2)))

print(f"\nPeak chunks (clamped to {MAX_PEAK_CHUNK_S}s max):")
for cs, ce in peak_chunks:
    print(f"  [{cs}-{ce}s] ({ce-cs:.2f}s)")

# --------------------------------------------------
# FILL GAPS
# --------------------------------------------------

def fill_gap(gs, ge):
    gap_chunks = []
    if (ge - gs) < MIN_GAP_CHUNK_S:
        return gap_chunks
    t = gs
    while t < ge:
        chunk_end = min(t + MAX_GAP_CHUNK_S, ge)
        gap_chunks.append((round(t, 2), round(chunk_end, 2), "gap"))
        t = chunk_end
    return gap_chunks

all_chunks = []

if not peak_chunks:
    all_chunks.extend(fill_gap(0.0, duration))
else:
    # Gap before first peak
    all_chunks.extend(fill_gap(0.0, peak_chunks[0][0]))

    # Interleave peak chunks and gaps
    for idx, (cs, ce) in enumerate(peak_chunks):
        all_chunks.append((cs, ce, "peak"))

        # Gap after this peak chunk
        next_start = peak_chunks[idx + 1][0] if idx + 1 < len(peak_chunks) else duration
        all_chunks.extend(fill_gap(ce, next_start))

all_chunks.sort(key=lambda x: x[0])

peak_count = sum(1 for _, _, t in all_chunks if t == "peak")
gap_count  = sum(1 for _, _, t in all_chunks if t == "gap")

print(f"\n--- CHUNKS ---")
print(f"Total chunks: {len(all_chunks)} (peak: {peak_count}, gap: {gap_count})")
for cs, ce, ctype in all_chunks:
    print(f"  [{cs}-{ce}s] {ctype} ({ce-cs:.2f}s)")

# --------------------------------------------------
# FRAME SAMPLING — 4 frames: start, top 2 importance, end
# --------------------------------------------------

def sample_frames(start, end):
    indices = [i for i, (ts, _, _, _) in enumerate(frames) if start <= ts <= end]

    if not indices:
        return []

    if len(indices) <= 2:
        return [Image.fromarray(frames[i][3]) for i in indices]

    first_idx = indices[0]
    last_idx  = indices[-1]

    middle_candidates = indices[1:-1]
    if len(middle_candidates) >= 2:
        sorted_mid = sorted(middle_candidates, key=lambda i: importance[i], reverse=True)
        peak1_idx = sorted_mid[0]
        peak2_idx = sorted_mid[1]
    elif len(middle_candidates) == 1:
        peak1_idx = middle_candidates[0]
        peak2_idx = None
    else:
        peak1_idx = None
        peak2_idx = None

    selected_set = [first_idx]
    peaks_sel = sorted([p for p in [peak1_idx, peak2_idx] if p is not None])
    for p in peaks_sel:
        if p not in selected_set:
            selected_set.append(p)
    if last_idx not in selected_set:
        selected_set.append(last_idx)

    selected_set.sort()
    return [Image.fromarray(frames[i][3]) for i in selected_set]

# --------------------------------------------------
# PROMPT
# --------------------------------------------------

SYSTEM_PROMPT = """
You are a dashcam footage analyst.

STRICT RULES:
- Base your description ONLY on what is directly visible in the provided frames.
- Do NOT carry forward objects or events from the previous scene unless they are still visible in the current frames.
- If the previous scene described something that is NOT visible in the current frames, do not mention it.
- Do NOT speculate or infer. If you cannot see it clearly, do not state it.

Left = driver's left (oncoming traffic side).
Right = driver's right (shoulder/curb side).

Scan the frame systematically — foreground to background, left to right:
- vehicles: type, color, position, direction of travel, and whether moving or stopped
- people: anyone visible — walking on road, standing on shoulder, riding in/on vehicles ahead, inside truck beds — note their apparent role if visible (police, military, worker, civilian)
- road infrastructure: cones, barriers, checkpoints, stop signs, roadblocks, lane markings
- abnormal events: swerving, lane departure, wrong-way driving, sudden stops, emergency lights, smoke, debris
"""

CHUNK_PROMPT = """
Previous scene context (for orientation only — do NOT repeat it):
{prev}

Current segment: {start}-{end}s.

You are shown up to 4 frames from this segment: the start, two key moments, and the end.

Examine each frame carefully. Pay specific attention to:
- any vehicle changing position, direction, or leaving the paved road
- any vehicle crossing a lane line or moving onto the shoulder or grass
- any person visible anywhere — on the road, on the shoulder, or on/in any vehicle
- any vehicle braking hard, stopping suddenly, or showing emergency lights
- any infrastructure like checkpoints, barriers, cones, or signs

Write one to two concise sentences. First sentence: the main action or change between frames. Second sentence (if needed): notable people, their roles, and road infrastructure.
"""

# --------------------------------------------------
# CONTEXT BUILDER
# --------------------------------------------------

def build_context(text, start, end):
    short = text[:MEMORY_CHARS].strip()
    if len(text) > MEMORY_CHARS and " " in short:
        short = short.rsplit(" ", 1)[0]
    return f"{start}-{end}s: {short}"

# --------------------------------------------------
# MODEL INFERENCE
# --------------------------------------------------

def analyze_chunk(frame_imgs, start, end, prev):
    prompt = CHUNK_PROMPT.format(start=start, end=end, prev=prev)

    image_msgs = [{"type": "image", "image": f} for f in frame_imgs]

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": image_msgs + [{"type": "text", "text": prompt}]}
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, _ = process_vision_info(messages)

    inputs = processor(
        text=[text],
        images=image_inputs,
        padding=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_TOKENS,
            temperature=0,
            do_sample=False
        )

    trimmed = output[:, inputs["input_ids"].shape[1]:]
    result  = processor.batch_decode(trimmed, skip_special_tokens=True)[0]

    del inputs, output
    gc.collect()
    torch.cuda.empty_cache()

    return result.strip()

# --------------------------------------------------
# RUN PIPELINE
# --------------------------------------------------

timeline     = []
prev_context = "start of video"

for i, (start, end, chunk_type) in enumerate(all_chunks):
    label = "PEAK" if chunk_type == "peak" else "GAP"
    print(f"Chunk {i+1}/{len(all_chunks)} [{start}-{end}s] {label} ({end-start:.2f}s)")

    frame_imgs = sample_frames(start, end)
    if frame_imgs:
        text         = analyze_chunk(frame_imgs, start, end, prev_context)
        prev_context = build_context(text, start, end)
        timeline.append((start, end, chunk_type, text))

# --------------------------------------------------
# PRINT TIMELINE
# --------------------------------------------------

print("\nTIMELINE\n")

for s, e, ct, t in timeline:
    tag = " [PEAK]" if ct == "peak" else ""
    print(f"[{s}-{e}s]{tag} {t}")

print(f"\nTotal chunks: {len(all_chunks)} (peak: {peak_count}, gap: {gap_count})")
print(f"Total runtime: {time.time() - start_time:.1f}s")

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

Video duration: 31.66s  |  FPS: 29.97002997002997  |  Frames: 949
Frames extracted: 316

--- PEAK DETECTION ---
Importance threshold (p75): 11.7387
Min peak distance: 1.5s (12 frames)
Peaks found: 9
  Peak at 4.30s
  Peak at 9.71s
  Peak at 11.11s
  Peak at 12.41s
  Peak at 14.41s
  Peak at 16.02s
  Peak at 19.02s
  Peak at 20.82s
  Peak at 24.42s

Peak chunks (clamped to 4.0s max):
  [2.8-6.3s] (3.50s)
  [7.71-10.41s] (2.70s)
  [10.41-11.76s] (1.35s)
  [11.76-13.41s] (1.65s)
  [13.41-15.22s] (1.81s)
  [15.22-17.52s] (2.30s)
  [17.52-19.92s] (2.40s)
  [19.92-22.62s] (2.70s)
  [22.62-25.92s] (3.30s)

--- CHUNKS ---
Total chunks: 12 (peak: 9, gap: 3)
  [0.0-2.8s] gap (2.80s)
  [2.8-6.3s] peak (3.50s)
  [6.3-7.71s] gap (1.41s)
  [7.71-10.41s] peak (2.70s)
  [10.41-11.76s] peak (1.35s)
  [11.76-13.41s] peak (1.65s)
  [13.41-15.22s] peak (1.81s)
  [15.22-17.52s] peak (2.30s)
  [17.52-19.92s] peak (2.40s)
  [19.92-22.62s] peak (2.70s)
  [22.62-25.92s] peak (3.30s)
  [25.92-31.66s] gap (5.74s

Trying with dynamic peaks in the video

In [30]:
# ===== Dashcam Analyzer v28 — Content-Driven Chunking =====
# Core idea: let signal peaks define chunk boundaries instead of fixed time windows.
# Peak regions get tight, detailed chunks. Calm regions get single summary chunks.
# No event/normal classification. No thresholds. No percentiles.

import torch
import cv2
import numpy as np
import gc
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText
from qwen_vl_utils import process_vision_info
from scipy.signal import find_peaks
import time

VIDEO_PATH = "/content/military.mp4"
MODEL_ID   = "Qwen/Qwen3-VL-8B-Instruct"

# --------------------------------------------------
# Load model
# --------------------------------------------------

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

# --------------------------------------------------
# CONFIG — each parameter explained
# --------------------------------------------------

start_time = time.time()

# FLOW_FPS: How many frames per second to extract from the video.
# Higher = more frames to analyze = better signal resolution but slower preprocessing.
# 8 is a good balance — enough to catch fast events without exploding memory.
FLOW_FPS = 8

# MAX_TOKENS: Maximum tokens the model can generate per chunk description.
# 120 gives room for 2 concise sentences without being wasteful.
MAX_TOKENS = 120

# MEMORY_CHARS: How many characters of the previous description to carry forward
# as context for the next chunk. Too low = model loses continuity (forgets a truck
# went off-road). Too high = wastes input tokens on stale context.
MEMORY_CHARS = 100

# ROLLING_WINDOW: Number of previous frames used to compute the scene delta signal.
# Scene delta compares the current frame against the average of the last N frames.
# 8 frames at 8 FPS = 1 second lookback. Catches changes that develop over ~1s.
ROLLING_WINDOW = 8

# PEAK_MIN_DISTANCE_S: Minimum seconds between detected peaks.
# Prevents the system from treating every frame of a single event as a separate peak.
# 1.5s means "after finding a peak, ignore the next 1.5s before looking for another."
# Too low = clusters of redundant peaks. Too high = misses back-to-back events.
PEAK_MIN_DISTANCE_S = 1.5

# PEAK_CHUNK_PAD_S: How many seconds to extend a chunk on each side of a peak.
# A peak at t=10s with pad=1.5 creates a chunk from 8.5s to 11.5s.
# This ensures the model sees context before and after the peak moment.
# Too low = model misses the lead-up to an event. Too high = chunks get bloated.
PEAK_CHUNK_PAD_S = 1.5

# MAX_CHUNK_DURATION_S: Maximum allowed duration for any single gap-fill chunk.
# Even in a 60s static stretch, no single chunk will exceed this.
MAX_CHUNK_DURATION_S = 5.0

# MIN_GAP_CHUNK_S: Minimum gap duration worth describing.
# If the gap between two peak-chunks is shorter than this, just skip it —
# the overlapping peak chunks already cover it. Avoids tiny useless chunks.
MIN_GAP_CHUNK_S = 1.0

# PEAK_PERCENTILE: What percentile of the importance signal counts as a peak.
# find_peaks uses a height threshold — we set it to this percentile of all
# importance values. 75 means "only frames in the top 25% of activity can be peaks."
# Lower = more peaks = more chunks = better detail but slower.
# Higher = fewer peaks = faster but might miss subtle events.
PEAK_PERCENTILE = 75

# --------------------------------------------------
# VIDEO INFO
# --------------------------------------------------

cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise RuntimeError(f"Could not open video: {VIDEO_PATH}")

native_fps   = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

if native_fps == 0:
    raise RuntimeError("Could not read FPS — check video file format/codec.")

duration = total_frames / native_fps
print(f"Video duration: {duration:.2f}s  |  FPS: {native_fps}  |  Frames: {total_frames}")

# --------------------------------------------------
# FRAME EXTRACTION
# --------------------------------------------------

frame_interval = max(1, int(native_fps / FLOW_FPS))
frames         = []
frame_idx      = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break
    if frame_idx % frame_interval == 0:
        ts         = frame_idx / native_fps
        rgb        = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        gray       = cv2.cvtColor(cv2.resize(frame, (640, 360)), cv2.COLOR_BGR2GRAY)
        brightness = float(gray.mean())
        frames.append((ts, gray, brightness, rgb))
    frame_idx += 1

cap.release()
print(f"Frames extracted: {len(frames)}")

frame_timestamps = np.array([f[0] for f in frames])

# --------------------------------------------------
# SIGNAL COMPUTATION
# --------------------------------------------------

# Signal 1+2: Flow variance + forward flow collapse
flow_variance_scores = [0.0]
flow_collapse_scores = [0.0]
prev_forward_flow    = None

for i in range(1, len(frames)):
    prev_gray = frames[i-1][1]
    curr_gray = frames[i][1]

    flow = cv2.calcOpticalFlowFarneback(
        prev_gray, curr_gray, None,
        0.5, 3, 15, 3, 5, 1.2, 0
    )

    mag, _        = cv2.cartToPolar(flow[..., 0], flow[..., 1])
    flow_variance = float(np.std(mag))
    flow_variance_scores.append(flow_variance)

    forward_flow = float(np.mean(np.abs(flow[..., 1])))
    collapse     = max(0.0, prev_forward_flow - forward_flow) if prev_forward_flow is not None else 0.0
    flow_collapse_scores.append(collapse)
    prev_forward_flow = forward_flow

flow_var_arr      = np.array(flow_variance_scores)
flow_collapse_arr = np.array(flow_collapse_scores)

# Signal 3: Brightness second derivative
brightness_arr    = np.array([f[2] for f in frames])
d1                = np.abs(np.diff(brightness_arr))
d2                = np.abs(np.diff(d1))
brightness_scores = np.concatenate([[0, 0], d2])

# Signal 4: Scene delta
scene_delta_scores = []

for i in range(len(frames)):
    curr_gray    = frames[i][1].astype(np.float32)
    window_start = max(0, i - ROLLING_WINDOW)
    window_grays = np.array([frames[j][1].astype(np.float32) for j in range(window_start, i)])

    if len(window_grays) == 0:
        scene_delta_scores.append(0.0)
    else:
        rolling_mean = window_grays.mean(axis=0)
        scene_delta_scores.append(float(np.mean(np.abs(curr_gray - rolling_mean))))

scene_delta_arr = np.array(scene_delta_scores)

# Combined importance score per frame
importance = flow_var_arr + flow_collapse_arr + brightness_scores + scene_delta_arr

# --------------------------------------------------
# STEP 2: FIND PEAKS
# --------------------------------------------------

# Minimum distance between peaks in frames
min_distance_frames = max(1, int(PEAK_MIN_DISTANCE_S * FLOW_FPS))

# Height threshold: only frames above this percentile qualify as peaks
height_threshold = float(np.percentile(importance, PEAK_PERCENTILE))

peak_indices, peak_props = find_peaks(
    importance,
    height=height_threshold,
    distance=min_distance_frames
)

peak_times = frame_timestamps[peak_indices]

print(f"\n--- PEAK DETECTION ---")
print(f"Importance threshold (p{PEAK_PERCENTILE}): {height_threshold:.4f}")
print(f"Min peak distance: {PEAK_MIN_DISTANCE_S}s ({min_distance_frames} frames)")
print(f"Peaks found: {len(peak_indices)}")
for pt in peak_times:
    print(f"  Peak at {pt:.2f}s")

# --------------------------------------------------
# STEP 3: BUILD CHUNKS AROUND PEAKS (midpoint splitting)
# --------------------------------------------------

# Each peak owns the space from the midpoint between it and the previous peak
# to the midpoint between it and the next peak. The first peak extends back
# by PEAK_CHUNK_PAD_S, and the last peak extends forward by PEAK_CHUNK_PAD_S.
# This guarantees non-overlapping chunks with no degenerate widths.

merged_chunks = []

if len(peak_times) > 0:
    for i, pt in enumerate(peak_times):
        # Left boundary: midpoint to previous peak, or peak - pad for first peak
        if i == 0:
            cs = max(0.0, pt - PEAK_CHUNK_PAD_S)
        else:
            cs = (peak_times[i - 1] + pt) / 2.0

        # Right boundary: midpoint to next peak, or peak + pad for last peak
        if i == len(peak_times) - 1:
            ce = min(duration, pt + PEAK_CHUNK_PAD_S)
        else:
            ce = (pt + peak_times[i + 1]) / 2.0

        merged_chunks.append((round(cs, 2), round(ce, 2)))

print(f"Peak chunks: {len(merged_chunks)}")
for cs, ce in merged_chunks:
    print(f"  [{cs}-{ce}s] ({ce-cs:.2f}s)")

# --------------------------------------------------
# STEP 4: FILL GAPS
# --------------------------------------------------

def fill_gap(gs, ge):
    """Create gap-fill chunks for the interval [gs, ge]."""
    gap_chunks = []
    if (ge - gs) < MIN_GAP_CHUNK_S:
        return gap_chunks
    t = gs
    while t < ge:
        chunk_end = min(t + MAX_CHUNK_DURATION_S, ge)
        gap_chunks.append((round(t, 2), round(chunk_end, 2), "gap"))
        t = chunk_end
    return gap_chunks

all_chunks = []

if not merged_chunks:
    # No peaks at all — entire video is gap chunks
    all_chunks.extend(fill_gap(0.0, duration))
else:
    # Gap before first peak
    all_chunks.extend(fill_gap(0.0, merged_chunks[0][0]))

    # Interleave peak chunks and gaps
    for idx, (cs, ce) in enumerate(merged_chunks):
        all_chunks.append((cs, ce, "peak"))

        # Gap after this peak chunk
        next_start = merged_chunks[idx + 1][0] if idx + 1 < len(merged_chunks) else duration
        all_chunks.extend(fill_gap(ce, next_start))

# Sort by start time
all_chunks.sort(key=lambda x: x[0])

peak_count = sum(1 for _, _, t in all_chunks if t == "peak")
gap_count  = sum(1 for _, _, t in all_chunks if t == "gap")

print(f"\n--- CHUNKS ---")
print(f"Total chunks: {len(all_chunks)} (peak: {peak_count}, gap: {gap_count})")
for cs, ce, ctype in all_chunks:
    print(f"  [{cs}-{ce}s] {ctype}")

# --------------------------------------------------
# FRAME SAMPLING — 4 frames: start, top 2 peaks, end
# --------------------------------------------------

def sample_frames(start, end):
    """Pick up to 4 frames: first, two highest-importance, last."""
    indices = [i for i, (ts, _, _, _) in enumerate(frames) if start <= ts <= end]

    if not indices:
        return []

    if len(indices) <= 2:
        return [Image.fromarray(frames[i][3]) for i in indices]

    first_idx = indices[0]
    last_idx  = indices[-1]

    # Top 2 peak frames by importance, excluding first and last
    middle_candidates = indices[1:-1]
    if len(middle_candidates) >= 2:
        sorted_mid = sorted(middle_candidates, key=lambda i: importance[i], reverse=True)
        peak1_idx = sorted_mid[0]
        peak2_idx = sorted_mid[1]
    elif len(middle_candidates) == 1:
        peak1_idx = middle_candidates[0]
        peak2_idx = None
    else:
        peak1_idx = None
        peak2_idx = None

    # Build selection in temporal order
    selected_set = [first_idx]
    peaks_sel = sorted([p for p in [peak1_idx, peak2_idx] if p is not None])
    for p in peaks_sel:
        if p not in selected_set:
            selected_set.append(p)
    if last_idx not in selected_set:
        selected_set.append(last_idx)

    selected_set.sort()

    return [Image.fromarray(frames[i][3]) for i in selected_set]

# --------------------------------------------------
# PROMPTS
# --------------------------------------------------

SYSTEM_PROMPT = """
You are a dashcam footage analyst.

STRICT RULES:
- Base your description ONLY on what is directly visible in the provided frames.
- Do NOT carry forward objects or events from the previous scene unless they are still visible in the current frames.
- If the previous scene described something that is NOT visible in the current frames, do not mention it.
- Do NOT speculate or infer. If you cannot see it clearly, do not state it.

Left = driver's left (oncoming traffic side).
Right = driver's right (shoulder/curb side).

Scan the frame systematically — foreground to background, left to right:
- vehicles: type, color, position, direction of travel, and whether moving or stopped
- people: anyone visible — walking on road, standing on shoulder, riding in/on vehicles ahead, inside truck beds — note their apparent role if visible (police, military, worker, civilian)
- road infrastructure: cones, barriers, checkpoints, stop signs, roadblocks, lane markings
- abnormal events: swerving, lane departure, wrong-way driving, sudden stops, emergency lights, smoke, debris
"""

CHUNK_PROMPT = """
Previous scene context (for orientation only — do NOT repeat it):
{prev}

Current segment: {start}-{end}s.

You are shown up to 4 frames from this segment: the start, two key moments, and the end.

Examine each frame carefully. Pay specific attention to:
- any vehicle changing position, direction, or leaving the paved road
- any vehicle crossing a lane line or moving onto the shoulder or grass
- any person visible anywhere — on the road, on the shoulder, or on/in any vehicle
- any vehicle braking hard, stopping suddenly, or showing emergency lights
- any infrastructure like checkpoints, barriers, cones, or signs

Write one to two concise sentences. First sentence: the main action or change between frames. Second sentence (if needed): notable people, their roles, and road infrastructure.
"""

# --------------------------------------------------
# CONTEXT BUILDER
# --------------------------------------------------

def build_context(text, start, end):
    short = text[:MEMORY_CHARS].strip()
    if len(text) > MEMORY_CHARS and " " in short:
        short = short.rsplit(" ", 1)[0]
    return f"{start}-{end}s: {short}"

# --------------------------------------------------
# MODEL INFERENCE
# --------------------------------------------------

def analyze_chunk(frame_imgs, start, end, prev):
    prompt = CHUNK_PROMPT.format(start=start, end=end, prev=prev)

    image_msgs = [{"type": "image", "image": f} for f in frame_imgs]

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": image_msgs + [{"type": "text", "text": prompt}]}
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, _ = process_vision_info(messages)

    inputs = processor(
        text=[text],
        images=image_inputs,
        padding=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_TOKENS,
            temperature=0,
            do_sample=False
        )

    trimmed = output[:, inputs["input_ids"].shape[1]:]
    result  = processor.batch_decode(trimmed, skip_special_tokens=True)[0]

    del inputs, output
    gc.collect()
    torch.cuda.empty_cache()

    return result.strip()

# --------------------------------------------------
# RUN PIPELINE
# --------------------------------------------------

timeline     = []
prev_context = "start of video"

for i, (start, end, chunk_type) in enumerate(all_chunks):
    label = "PEAK" if chunk_type == "peak" else "GAP"
    print(f"Chunk {i+1}/{len(all_chunks)} [{start}-{end}s] {label}")

    frame_imgs = sample_frames(start, end)
    if frame_imgs:
        text         = analyze_chunk(frame_imgs, start, end, prev_context)
        prev_context = build_context(text, start, end)
        timeline.append((start, end, chunk_type, text))

# --------------------------------------------------
# PRINT TIMELINE
# --------------------------------------------------

print("\nTIMELINE\n")

for s, e, ct, t in timeline:
    tag = " [PEAK]" if ct == "peak" else ""
    print(f"[{s}-{e}s]{tag} {t}")

print(f"\nTotal chunks: {len(all_chunks)} (peak: {peak_count}, gap: {gap_count})")
print(f"Total runtime: {time.time() - start_time:.1f}s")

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

Video duration: 97.10s  |  FPS: 30.0  |  Frames: 2913
Frames extracted: 970

--- PEAK DETECTION ---
Importance threshold (p75): 10.5878
Min peak distance: 1.5s (12 frames)
Peaks found: 18
  Peak at 36.10s
  Peak at 37.90s
  Peak at 39.20s
  Peak at 40.60s
  Peak at 42.70s
  Peak at 46.00s
  Peak at 54.40s
  Peak at 69.50s
  Peak at 70.80s
  Peak at 72.30s
  Peak at 73.90s
  Peak at 76.90s
  Peak at 86.90s
  Peak at 88.30s
  Peak at 89.70s
  Peak at 91.60s
  Peak at 93.60s
  Peak at 96.00s
Peak chunks: 18
  [34.6-37.0s] (2.40s)
  [37.0-38.55s] (1.55s)
  [38.55-39.9s] (1.35s)
  [39.9-41.65s] (1.75s)
  [41.65-44.35s] (2.70s)
  [44.35-50.2s] (5.85s)
  [50.2-61.95s] (11.75s)
  [61.95-70.15s] (8.20s)
  [70.15-71.55s] (1.40s)
  [71.55-73.1s] (1.55s)
  [73.1-75.4s] (2.30s)
  [75.4-81.9s] (6.50s)
  [81.9-87.6s] (5.70s)
  [87.6-89.0s] (1.40s)
  [89.0-90.65s] (1.65s)
  [90.65-92.6s] (1.95s)
  [92.6-94.8s] (2.20s)
  [94.8-97.1s] (2.30s)

--- CHUNKS ---
Total chunks: 25 (peak: 18, gap: 7)
  [0.0-5.

Trying out something simpler, normal chunks didn't have enough context, trying without classifying evnets and using always a set number of frames in a chunk - first, last and middle = decided by signals (what's most important to be middle)

In [ ]:
# ===== Dashcam Analyzer v26 — Uniform 3-Frame Pipeline =====
# Simplified from v25:
#   - No event/normal classification. Every chunk treated equally.
#   - 3 frames per chunk: start, peak-signal, end.
#   - Signals used ONLY for picking the best middle frame, not for flagging.
#   - Single prompt tier for all chunks.
#   - MEMORY_CHARS = 100 for better continuity.

import torch
import cv2
import numpy as np
import gc
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText
from qwen_vl_utils import process_vision_info
import time

VIDEO_PATH = "/content/military.mp4"
MODEL_ID   = "Qwen/Qwen3-VL-8B-Instruct"

# --------------------------------------------------
# Load model
# --------------------------------------------------

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

# --------------------------------------------------
# CONFIG
# --------------------------------------------------

start_time     = time.time()
FLOW_FPS       = 8
MAX_TOKENS     = 100
MEMORY_CHARS   = 100
ROLLING_WINDOW = 8

# --------------------------------------------------
# VIDEO INFO
# --------------------------------------------------

cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise RuntimeError(f"Could not open video: {VIDEO_PATH}")

native_fps   = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

if native_fps == 0:
    raise RuntimeError("Could not read FPS — check video file format/codec.")

duration = total_frames / native_fps
print(f"Video duration: {duration:.2f}s  |  FPS: {native_fps}  |  Frames: {total_frames}")

# --------------------------------------------------
# FRAME EXTRACTION
# --------------------------------------------------

frame_interval = max(1, int(native_fps / FLOW_FPS))
frames         = []
frame_idx      = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break
    if frame_idx % frame_interval == 0:
        ts         = frame_idx / native_fps
        rgb        = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        gray       = cv2.cvtColor(cv2.resize(frame, (640, 360)), cv2.COLOR_BGR2GRAY)
        brightness = float(gray.mean())
        frames.append((ts, gray, brightness, rgb))
    frame_idx += 1

cap.release()
print(f"Frames extracted: {len(frames)}")

# --------------------------------------------------
# SIGNAL COMPUTATION (for frame selection only)
# --------------------------------------------------

# Signal 1+2: Flow variance + forward flow collapse
flow_variance_scores = [0.0]
flow_collapse_scores = [0.0]
prev_forward_flow    = None

for i in range(1, len(frames)):
    prev_gray = frames[i-1][1]
    curr_gray = frames[i][1]

    flow = cv2.calcOpticalFlowFarneback(
        prev_gray, curr_gray, None,
        0.5, 3, 15, 3, 5, 1.2, 0
    )

    mag, _        = cv2.cartToPolar(flow[..., 0], flow[..., 1])
    flow_variance = float(np.std(mag))
    flow_variance_scores.append(flow_variance)

    forward_flow = float(np.mean(np.abs(flow[..., 1])))
    collapse     = max(0.0, prev_forward_flow - forward_flow) if prev_forward_flow is not None else 0.0
    flow_collapse_scores.append(collapse)
    prev_forward_flow = forward_flow

flow_var_arr      = np.array(flow_variance_scores)
flow_collapse_arr = np.array(flow_collapse_scores)

# Signal 3: Brightness second derivative
brightness_arr    = np.array([f[2] for f in frames])
d1                = np.abs(np.diff(brightness_arr))
d2                = np.abs(np.diff(d1))
brightness_scores = np.concatenate([[0, 0], d2])

# Signal 4: Scene delta
scene_delta_scores = []

for i in range(len(frames)):
    curr_gray    = frames[i][1].astype(np.float32)
    window_start = max(0, i - ROLLING_WINDOW)
    window_grays = np.array([frames[j][1].astype(np.float32) for j in range(window_start, i)])

    if len(window_grays) == 0:
        scene_delta_scores.append(0.0)
    else:
        rolling_mean = window_grays.mean(axis=0)
        scene_delta_scores.append(float(np.mean(np.abs(curr_gray - rolling_mean))))

scene_delta_arr = np.array(scene_delta_scores)

# Combined importance score per frame (raw sum, used only for picking best middle frame)
importance = flow_var_arr + flow_collapse_arr + brightness_scores + scene_delta_arr

# --------------------------------------------------
# DYNAMIC CHUNK DURATION
# --------------------------------------------------

median_flow = float(np.median(flow_var_arr))
CHUNK_DURATION = float(np.clip(3.0 - 0.3 * median_flow, 1.5, 3.0))
CHUNK_DURATION = round(CHUNK_DURATION * 4) / 4
OVERLAP = round(CHUNK_DURATION * 0.25, 2)

print(f"\n--- CONFIG ---")
print(f"Median flow:      {median_flow:.3f}")
print(f"Chunk duration:   {CHUNK_DURATION}s  |  Overlap: {OVERLAP}s")
print(f"--- END CONFIG ---\n")

# --------------------------------------------------
# BUILD CHUNKS
# --------------------------------------------------

step         = CHUNK_DURATION - OVERLAP
chunk_starts = np.arange(0, duration - OVERLAP, step)
chunks       = [(round(cs, 2), round(min(cs + CHUNK_DURATION, duration), 2)) for cs in chunk_starts]

print(f"Total chunks: {len(chunks)}")

# --------------------------------------------------
# FRAME SAMPLING — 3 frames: start, peak, end
# --------------------------------------------------

frame_timestamps = np.array([f[0] for f in frames])

def sample_frames(start, end):
    """Pick 3 frames: first in chunk, highest-importance in chunk, last in chunk."""
    indices = [i for i, (ts, _, _, _) in enumerate(frames) if start <= ts <= end]

    if not indices:
        return []

    if len(indices) == 1:
        return [Image.fromarray(frames[indices[0]][3])]

    if len(indices) == 2:
        return [Image.fromarray(frames[indices[0]][3]),
                Image.fromarray(frames[indices[1]][3])]

    first_idx = indices[0]
    last_idx  = indices[-1]

    # Best middle frame: highest importance, excluding first and last
    middle_candidates = indices[1:-1]
    if middle_candidates:
        peak_idx = max(middle_candidates, key=lambda i: importance[i])
    else:
        # Only 3 frames total, middle is the only option
        peak_idx = indices[1]

    # Deduplicate (if peak happens to be first or last)
    selected = list(dict.fromkeys([first_idx, peak_idx, last_idx]))

    return [Image.fromarray(frames[i][3]) for i in selected]

# --------------------------------------------------
# PROMPT
# --------------------------------------------------

SYSTEM_PROMPT = """
You are a dashcam footage analyst.

STRICT RULES:
- Base your description ONLY on what is directly visible in the provided frames.
- Do NOT carry forward objects or events from the previous scene unless they are still visible in the current frames.
- If the previous scene described something that is NOT visible in the current frames, do not mention it.
- Do NOT speculate or infer. If you cannot see it clearly, do not state it.

Left = driver's left (oncoming traffic side).
Right = driver's right (shoulder/curb side).

Look for:
- vehicles and their positions, especially any leaving the lane or road
- pedestrians appearing in or near the road and identify their occupation if visible, such as police officer, military personnel, pedestrian
- road infrastructure: cones, barriers, checkpoints, stop signs, roadblocks
- abnormal events: swerving, lane departure, wrong-way vehicles, sudden stops, emergency lights, smoke
"""

CHUNK_PROMPT = """
Previous scene context (for orientation only — do NOT repeat it):
{prev}

Current segment: {start}-{end}s.

You are shown 3 frames from this segment: the start, a key middle moment, and the end.

Look carefully at ALL vehicles and people across the frames. Pay attention to:
- any change in vehicle position or direction between frames
- any vehicle leaving the road, drifting, swerving, or stopping
- any person appearing in or near the road
- any new infrastructure: barriers, cones, signs, checkpoints

Describe what is happening in ONE concise sentence. If something changed between the start and end frames, describe the change.
"""

# --------------------------------------------------
# CONTEXT BUILDER
# --------------------------------------------------

def build_context(text, start, end):
    short = text[:MEMORY_CHARS].strip()
    if len(text) > MEMORY_CHARS and " " in short:
        short = short.rsplit(" ", 1)[0]
    return f"{start}-{end}s: {short}"

# --------------------------------------------------
# MODEL INFERENCE
# --------------------------------------------------

def analyze_chunk(frame_imgs, start, end, prev):
    prompt = CHUNK_PROMPT.format(start=start, end=end, prev=prev)

    image_msgs = [{"type": "image", "image": f} for f in frame_imgs]

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": image_msgs + [{"type": "text", "text": prompt}]}
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, _ = process_vision_info(messages)

    inputs = processor(
        text=[text],
        images=image_inputs,
        padding=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_TOKENS,
            temperature=0,
            do_sample=False
        )

    trimmed = output[:, inputs["input_ids"].shape[1]:]
    result  = processor.batch_decode(trimmed, skip_special_tokens=True)[0]

    del inputs, output
    gc.collect()
    torch.cuda.empty_cache()

    return result.strip()

# --------------------------------------------------
# RUN PIPELINE
# --------------------------------------------------

timeline     = []
prev_context = "start of video"

for i, (start, end) in enumerate(chunks):
    print(f"Chunk {i+1}/{len(chunks)} [{start}-{end}s]")

    frame_imgs = sample_frames(start, end)
    if frame_imgs:
        text         = analyze_chunk(frame_imgs, start, end, prev_context)
        prev_context = build_context(text, start, end)
        timeline.append((start, end, text))

# --------------------------------------------------
# PRINT TIMELINE
# --------------------------------------------------

print("\nTIMELINE\n")

for s, e, t in timeline:
    print(f"[{s}-{e}s] {t}")

print(f"\nTotal chunks: {len(chunks)}")
print(f"Total runtime: {time.time() - start_time:.1f}s")

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

Video duration: 97.10s  |  FPS: 30.0  |  Frames: 2913
Frames extracted: 970

--- CONFIG ---
Median flow:      1.410
Chunk duration:   2.5s  |  Overlap: 0.62s
--- END CONFIG ---

Total chunks: 52
Chunk 1/52 [0.0-2.5s]
Chunk 2/52 [1.88-4.38s]
Chunk 3/52 [3.76-6.26s]
Chunk 4/52 [5.64-8.14s]
Chunk 5/52 [7.52-10.02s]
Chunk 6/52 [9.4-11.9s]
Chunk 7/52 [11.28-13.78s]
Chunk 8/52 [13.16-15.66s]
Chunk 9/52 [15.04-17.54s]
Chunk 10/52 [16.92-19.42s]
Chunk 11/52 [18.8-21.3s]
Chunk 12/52 [20.68-23.18s]
Chunk 13/52 [22.56-25.06s]
Chunk 14/52 [24.44-26.94s]
Chunk 15/52 [26.32-28.82s]
Chunk 16/52 [28.2-30.7s]
Chunk 17/52 [30.08-32.58s]
Chunk 18/52 [31.96-34.46s]
Chunk 19/52 [33.84-36.34s]
Chunk 20/52 [35.72-38.22s]
Chunk 21/52 [37.6-40.1s]
Chunk 22/52 [39.48-41.98s]
Chunk 23/52 [41.36-43.86s]
Chunk 24/52 [43.24-45.74s]
Chunk 25/52 [45.12-47.62s]
Chunk 26/52 [47.0-49.5s]
Chunk 27/52 [48.88-51.38s]
Chunk 28/52 [50.76-53.26s]
Chunk 29/52 [52.64-55.14s]
Chunk 30/52 [54.52-57.02s]
Chunk 31/52 [56.4-58.9s]
C

Fixing the overfiring signal problem + trying ot fix teh normal batching problem where its losing a lot of context + dynamic params

In [ ]:
# ===== Dashcam Analyzer v24 — No Batch Grid (Option A) =====
# Change from v23: removed grid batching for normal chunks.
# Every chunk is now processed individually to avoid hallucinated
# disappearances caused by side-by-side frame comparison.

import torch
import cv2
import numpy as np
import gc
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText
from qwen_vl_utils import process_vision_info
import time

VIDEO_PATH = "/content/ukraine_road.mp4"
MODEL_ID   = "Qwen/Qwen3-VL-8B-Instruct"

# --------------------------------------------------
# Load model
# --------------------------------------------------

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

# --------------------------------------------------
# STATIC CONFIG
# --------------------------------------------------

start_time        = time.time()
FLOW_FPS          = 8
MAX_FRAMES_NORMAL = 1
MAX_FRAMES_EVENT  = 4
NORMAL_TOKENS     = 80
EVENT_TOKENS      = 120
MEMORY_CHARS      = 40
ROLLING_WINDOW    = 8

# --------------------------------------------------
# VIDEO INFO
# --------------------------------------------------

cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise RuntimeError(f"Could not open video: {VIDEO_PATH}")

native_fps   = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

if native_fps == 0:
    raise RuntimeError("Could not read FPS — check video file format/codec.")

duration = total_frames / native_fps
print(f"Video duration: {duration:.2f}s  |  FPS: {native_fps}  |  Frames: {total_frames}")

# --------------------------------------------------
# FRAME EXTRACTION
# --------------------------------------------------

frame_interval = max(1, int(native_fps / FLOW_FPS))
frames         = []
frame_idx      = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break
    if frame_idx % frame_interval == 0:
        ts         = frame_idx / native_fps
        rgb        = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        gray       = cv2.cvtColor(cv2.resize(frame, (640, 360)), cv2.COLOR_BGR2GRAY)
        brightness = float(gray.mean())
        frames.append((ts, gray, brightness, rgb))
    frame_idx += 1

cap.release()
print(f"Frames extracted: {len(frames)}")

# --------------------------------------------------
# SIGNAL 1+2: FLOW VARIANCE + FORWARD FLOW COLLAPSE
# --------------------------------------------------

flow_variance_scores = [0.0]
flow_collapse_scores = [0.0]
prev_forward_flow    = None

for i in range(1, len(frames)):
    prev_gray = frames[i-1][1]
    curr_gray = frames[i][1]

    flow = cv2.calcOpticalFlowFarneback(
        prev_gray, curr_gray, None,
        0.5, 3, 15, 3, 5, 1.2, 0
    )

    mag, _        = cv2.cartToPolar(flow[..., 0], flow[..., 1])
    flow_variance = float(np.std(mag))
    flow_variance_scores.append(flow_variance)

    forward_flow = float(np.mean(np.abs(flow[..., 1])))
    collapse     = max(0.0, prev_forward_flow - forward_flow) if prev_forward_flow is not None else 0.0
    flow_collapse_scores.append(collapse)
    prev_forward_flow = forward_flow

flow_var_arr      = np.array(flow_variance_scores)
flow_collapse_arr = np.array(flow_collapse_scores)

# --------------------------------------------------
# SIGNAL 3: BRIGHTNESS SECOND DERIVATIVE
# --------------------------------------------------

brightness_arr    = np.array([f[2] for f in frames])
d1                = np.abs(np.diff(brightness_arr))
d2                = np.abs(np.diff(d1))
brightness_scores = np.concatenate([[0, 0], d2])

# --------------------------------------------------
# SIGNAL 4: SCENE DELTA
# --------------------------------------------------

scene_delta_scores = []

for i in range(len(frames)):
    curr_gray    = frames[i][1].astype(np.float32)
    window_start = max(0, i - ROLLING_WINDOW)
    window_grays = np.array([frames[j][1].astype(np.float32) for j in range(window_start, i)])

    if len(window_grays) == 0:
        scene_delta_scores.append(0.0)
    else:
        rolling_mean = window_grays.mean(axis=0)
        scene_delta_scores.append(float(np.mean(np.abs(curr_gray - rolling_mean))))

scene_delta_arr = np.array(scene_delta_scores)

# --------------------------------------------------
# ADAPTIVE MULTIPLIER
# --------------------------------------------------

def adaptive_multiplier(arr, base_mult=2.0):
    cv = arr.std() / (arr.mean() + 1e-6)
    cv = min(cv, 1.0)
    return base_mult * (1 + 0.5 * cv)

# --------------------------------------------------
# DYNAMIC PARAMETERS
# --------------------------------------------------

mean_flow = float(np.mean(flow_var_arr))

if mean_flow < 1.5:
    CHUNK_DURATION = 2.5
    OVERLAP        = 0.5
else:
    CHUNK_DURATION = 2.0
    OVERLAP        = 0.75

CHUNK_DURATION = min(2.5, CHUNK_DURATION)

FLOW_VAR_STD_MULT      = adaptive_multiplier(flow_var_arr,      base_mult=2.0)
FLOW_COLLAPSE_STD_MULT = adaptive_multiplier(flow_collapse_arr, base_mult=2.0)
BRIGHTNESS_STD_MULT    = adaptive_multiplier(brightness_scores, base_mult=2.0)
SCENE_DELTA_STD_MULT   = adaptive_multiplier(scene_delta_arr,   base_mult=2.0)

print(f"\n--- DYNAMIC CONFIG ---")
print(f"Mean flow:          {mean_flow:.3f}")
print(f"Chunk duration:     {CHUNK_DURATION}s  |  Overlap: {OVERLAP}s")
print(f"Flow var mult:      {FLOW_VAR_STD_MULT:.3f}")
print(f"Flow collapse mult: {FLOW_COLLAPSE_STD_MULT:.3f}")
print(f"Brightness mult:    {BRIGHTNESS_STD_MULT:.3f}")
print(f"Scene delta mult:   {SCENE_DELTA_STD_MULT:.3f}")
print(f"--- END CONFIG ---\n")

# --------------------------------------------------
# COMPUTE THRESHOLDS
# --------------------------------------------------

flow_var_threshold      = flow_var_arr.mean()      + FLOW_VAR_STD_MULT      * flow_var_arr.std()
flow_collapse_threshold = flow_collapse_arr.mean() + FLOW_COLLAPSE_STD_MULT * flow_collapse_arr.std()
brightness_threshold    = brightness_scores.mean() + BRIGHTNESS_STD_MULT    * brightness_scores.std()
scene_delta_threshold   = scene_delta_arr.mean()   + SCENE_DELTA_STD_MULT   * scene_delta_arr.std()

print(f"Flow variance threshold:  {flow_var_threshold:.4f}")
print(f"Flow collapse threshold:  {flow_collapse_threshold:.4f}")
print(f"Brightness threshold:     {brightness_threshold:.4f}")
print(f"Scene delta threshold:    {scene_delta_threshold:.4f}")

# --------------------------------------------------
# BUILD CHUNKS
# --------------------------------------------------

step         = CHUNK_DURATION - OVERLAP
chunk_starts = np.arange(0, duration - OVERLAP, step)
chunks       = [(round(cs, 2), round(min(cs + CHUNK_DURATION, duration), 2)) for cs in chunk_starts]

# --------------------------------------------------
# FLAG EVENTS — dynamic MIN_SIGNALS
# --------------------------------------------------

frame_timestamps = np.array([f[0] for f in frames])

signal_count = (
    (flow_var_arr      >= flow_var_threshold).astype(int) +
    (flow_collapse_arr >= flow_collapse_threshold).astype(int) +
    (brightness_scores >= brightness_threshold).astype(int) +
    (scene_delta_arr   >= scene_delta_threshold).astype(int)
)

# First pass with MIN_SIGNALS=1
anomaly_mask = signal_count >= 1
chunk_flags  = []
for cs, ce in chunks:
    in_chunk = (frame_timestamps >= cs) & (frame_timestamps <= ce)
    chunk_flags.append(bool(np.any(anomaly_mask & in_chunk)))

event_ratio = sum(chunk_flags) / len(chunk_flags)
print(f"Event ratio (1-signal): {event_ratio:.2f}")

# Tighten to 2 signals if over 50% flagged
if event_ratio > 0.5:
    print("Too many events — tightening to MIN_SIGNALS=2")
    anomaly_mask = signal_count >= 2
    chunk_flags  = []
    for cs, ce in chunks:
        in_chunk = (frame_timestamps >= cs) & (frame_timestamps <= ce)
        chunk_flags.append(bool(np.any(anomaly_mask & in_chunk)))
    print(f"Event ratio (2-signal): {sum(chunk_flags)/len(chunk_flags):.2f}")

print(f"Event chunks: {sum(chunk_flags)} / {len(chunks)}")

# --------------------------------------------------
# FRAME SAMPLING
# --------------------------------------------------

def sample_frames(start, end, is_event):
    candidates = []

    for i, (ts, _, _, rgb) in enumerate(frames):
        if start <= ts <= end:
            importance = (
                flow_var_arr[i]
                + flow_collapse_arr[i]
                + brightness_scores[i]
                + scene_delta_arr[i]
            )
            candidates.append((i, importance, rgb))

    if not candidates:
        return []

    if not is_event:
        mid = candidates[len(candidates) // 2]
        return [Image.fromarray(mid[2])]

    candidates_sorted = sorted(candidates, key=lambda x: x[1], reverse=True)
    top_frames   = candidates_sorted[:2]
    start_frames = candidates[:1]
    end_frames   = candidates[-1:]

    unique = {}
    for i, score, rgb in (start_frames + top_frames + end_frames):
        unique[i] = rgb

    return [Image.fromarray(f) for f in list(unique.values())[:MAX_FRAMES_EVENT]]

# --------------------------------------------------
# PROMPTS
# --------------------------------------------------

SYSTEM_PROMPT = """
You are a dashcam footage analyst.

STRICT RULES:
- Base your description ONLY on what is directly visible in the provided frames.
- Do NOT carry forward objects or events from the previous scene unless they are still visible in the current frames.
- If the previous scene described something that is NOT visible in the current frames, do not mention it.
- Do NOT speculate or infer. If you cannot see it clearly, do not state it.

Left = driver's left (oncoming traffic side).
Right = driver's right (shoulder/curb side).

Look for:
- vehicles and their positions, especially any leaving the lane or road
- pedestrians appearing in or near the road and identify their occupation if visible, such as police officer, military personnel, pedestrian
- road infrastructure: cones, barriers, checkpoints, stop signs, roadblocks
- abnormal events: swerving, lane departure, wrong-way vehicles, sudden stops, emergency lights, smoke
"""

NORMAL_PROMPT = """
Previous scene context (for orientation only — do NOT repeat it):
{prev}

Current segment: {start}-{end}s.

Describe ONLY what is visible in the provided frame. One concise sentence.
"""

EVENT_PROMPT = """
Previous scene context (for orientation only — do NOT repeat it):
{prev}

Current segment: {start}-{end}s.

Something unusual may be occurring. Look carefully at ALL vehicles and people in the frame.

Pay specific attention to:
- any vehicle drifting, swerving, or leaving the paved road
- any vehicle crossing a lane line or moving onto the shoulder or grass
- any person appearing in or near the road and identify their occupation if visible
- any vehicle braking hard or stopping suddenly
- any infrastructure like checkpoints, barriers, or cones appearing

Do NOT mention anything from the previous scene that is not visible in these frames.
Write ONE concise sentence describing exactly what you see.
"""

# --------------------------------------------------
# CONTEXT BUILDER
# --------------------------------------------------

def build_context(text, start, end):
    short = text[:MEMORY_CHARS].strip()
    if len(text) > MEMORY_CHARS and " " in short:
        short = short.rsplit(" ", 1)[0]
    return f"{start}-{end}s: {short}"

# --------------------------------------------------
# MODEL INFERENCE
# --------------------------------------------------

def analyze_chunk(frame_imgs, start, end, is_event, prev):
    prompt = (EVENT_PROMPT if is_event else NORMAL_PROMPT).format(
        start=start, end=end, prev=prev
    )
    tokens = EVENT_TOKENS if is_event else NORMAL_TOKENS

    image_msgs = [{"type": "image", "image": f} for f in frame_imgs]

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": image_msgs + [{"type": "text", "text": prompt}]}
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, _ = process_vision_info(messages)

    inputs = processor(
        text=[text],
        images=image_inputs,
        padding=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=tokens,
            temperature=0,
            do_sample=False
        )

    trimmed = output[:, inputs["input_ids"].shape[1]:]
    result  = processor.batch_decode(trimmed, skip_special_tokens=True)[0]

    del inputs, output
    gc.collect()
    torch.cuda.empty_cache()

    return result.strip()

# --------------------------------------------------
# RUN PIPELINE — all chunks processed individually
# --------------------------------------------------

timeline     = []
prev_context = "start of video"

for i, (start, end) in enumerate(chunks):
    is_event = chunk_flags[i]

    print(f"Chunk {i+1}/{len(chunks)} [{start}-{end}s] event={is_event}")

    frame_imgs = sample_frames(start, end, is_event)
    if frame_imgs:
        text         = analyze_chunk(frame_imgs, start, end, is_event, prev_context)
        prev_context = build_context(text, start, end)
        timeline.append((start, end, is_event, text))

# --------------------------------------------------
# PRINT TIMELINE
# --------------------------------------------------

print("\nTIMELINE\n")

for s, e, ev, t in timeline:
    tag = " [EVENT]" if ev else ""
    print(f"[{s}-{e}s]{tag} {t}")

print(f"\nTotal runtime: {time.time() - start_time:.1f}s")

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

Video duration: 69.53s  |  FPS: 30.0  |  Frames: 2086
Frames extracted: 694

--- DYNAMIC CONFIG ---
Mean flow:          0.360
Chunk duration:     2.5s  |  Overlap: 0.5s
Flow var mult:      3.000
Flow collapse mult: 3.000
Brightness mult:    3.000
Scene delta mult:   2.866
--- END CONFIG ---

Flow variance threshold:  3.1448
Flow collapse threshold:  0.1168
Brightness threshold:     0.0905
Scene delta threshold:    2.5039
Event ratio (1-signal): 0.17
Event chunks: 6 / 35
Chunk 1/35 [0.0-2.5s] event=False
Chunk 2/35 [2.0-4.5s] event=True
Chunk 3/35 [4.0-6.5s] event=True
Chunk 4/35 [6.0-8.5s] event=True
Chunk 5/35 [8.0-10.5s] event=True
Chunk 6/35 [10.0-12.5s] event=False
Chunk 7/35 [12.0-14.5s] event=False
Chunk 8/35 [14.0-16.5s] event=False
Chunk 9/35 [16.0-18.5s] event=False
Chunk 10/35 [18.0-20.5s] event=False
Chunk 11/35 [20.0-22.5s] event=False
Chunk 12/35 [22.0-24.5s] event=False
Chunk 13/35 [24.0-26.5s] event=False
Chunk 14/35 [26.0-28.5s] event=False
Chunk 15/35 [28.0-30.5s] even

Fixing the left frame right frame problem with the prompt

In [ ]:
# ===== Dashcam Analyzer v23 + Grid Batching with Per-Chunk Descriptions =====

import torch
import cv2
import numpy as np
import gc
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText
from qwen_vl_utils import process_vision_info
import time

VIDEO_PATH = "/content/ukraine_road.mp4"
MODEL_ID   = "Qwen/Qwen3-VL-8B-Instruct"

# --------------------------------------------------
# Load model
# --------------------------------------------------

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

# --------------------------------------------------
# CONFIG
# --------------------------------------------------

start_time = time.time()
CHUNK_DURATION = 2.5
OVERLAP        = 0.5

FLOW_FPS          = 8
MAX_FRAMES_NORMAL = 1
MAX_FRAMES_EVENT  = 4

NORMAL_TOKENS     = 80      # increased to fit two sentences
EVENT_TOKENS      = 120
MEMORY_CHARS      = 40
NORMAL_BATCH_SIZE = 2

FLOW_VAR_STD_MULT      = 1.0
BRIGHTNESS_STD_MULT    = 1.8
SCENE_DELTA_STD_MULT   = 1.0
FLOW_COLLAPSE_STD_MULT = 1.5

# --------------------------------------------------
# VIDEO INFO
# --------------------------------------------------

cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise RuntimeError(f"Could not open video: {VIDEO_PATH}")

native_fps   = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

if native_fps == 0:
    raise RuntimeError("Could not read FPS — check video file format/codec.")

duration = total_frames / native_fps

print(f"Video duration: {duration:.2f}s  |  FPS: {native_fps}  |  Frames: {total_frames}")

# --------------------------------------------------
# FRAME EXTRACTION
# --------------------------------------------------

frame_interval = max(1, int(native_fps / FLOW_FPS))

frames    = []
frame_idx = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    if frame_idx % frame_interval == 0:
        ts         = frame_idx / native_fps
        rgb        = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        gray       = cv2.cvtColor(cv2.resize(frame, (640, 360)), cv2.COLOR_BGR2GRAY)
        brightness = float(gray.mean())
        frames.append((ts, gray, brightness, rgb))

    frame_idx += 1

cap.release()
print(f"Frames extracted: {len(frames)}")

# --------------------------------------------------
# SIGNAL 1+2: FLOW VARIANCE + FORWARD FLOW COLLAPSE
# --------------------------------------------------

flow_variance_scores = [0.0]
flow_collapse_scores = [0.0]
prev_forward_flow    = None

for i in range(1, len(frames)):
    prev_gray = frames[i-1][1]
    curr_gray = frames[i][1]

    flow = cv2.calcOpticalFlowFarneback(
        prev_gray, curr_gray, None,
        0.5, 3, 15, 3, 5, 1.2, 0
    )

    mag, _ = cv2.cartToPolar(flow[..., 0], flow[..., 1])

    flow_variance = float(np.std(mag))
    flow_variance_scores.append(flow_variance)

    forward_flow = float(np.mean(np.abs(flow[..., 1])))
    collapse     = max(0.0, prev_forward_flow - forward_flow) if prev_forward_flow is not None else 0.0
    flow_collapse_scores.append(collapse)
    prev_forward_flow = forward_flow

flow_var_arr      = np.array(flow_variance_scores)
flow_collapse_arr = np.array(flow_collapse_scores)

flow_var_threshold      = flow_var_arr.mean()      + FLOW_VAR_STD_MULT      * flow_var_arr.std()
flow_collapse_threshold = flow_collapse_arr.mean() + FLOW_COLLAPSE_STD_MULT * flow_collapse_arr.std()

print(f"Flow variance threshold:  {flow_var_threshold:.4f}")
print(f"Flow collapse threshold:  {flow_collapse_threshold:.4f}")

# --------------------------------------------------
# SIGNAL 3: BRIGHTNESS SECOND DERIVATIVE
# --------------------------------------------------

brightness_arr    = np.array([f[2] for f in frames])
d1                = np.abs(np.diff(brightness_arr))
d2                = np.abs(np.diff(d1))
brightness_scores = np.concatenate([[0, 0], d2])

brightness_threshold = brightness_scores.mean() + BRIGHTNESS_STD_MULT * brightness_scores.std()

print(f"Brightness threshold:     {brightness_threshold:.4f}")

# --------------------------------------------------
# SIGNAL 4: SCENE DELTA
# --------------------------------------------------

ROLLING_WINDOW = 8

scene_delta_scores = []

for i in range(len(frames)):
    curr_gray    = frames[i][1].astype(np.float32)
    window_start = max(0, i - ROLLING_WINDOW)
    window_grays = np.array([frames[j][1].astype(np.float32) for j in range(window_start, i)])

    if len(window_grays) == 0:
        scene_delta_scores.append(0.0)
    else:
        rolling_mean = window_grays.mean(axis=0)
        delta        = float(np.mean(np.abs(curr_gray - rolling_mean)))
        scene_delta_scores.append(delta)

scene_delta_arr       = np.array(scene_delta_scores)
scene_delta_threshold = scene_delta_arr.mean() + SCENE_DELTA_STD_MULT * scene_delta_arr.std()

print(f"Scene delta threshold:    {scene_delta_threshold:.4f}")

# --------------------------------------------------
# BUILD CHUNKS
# --------------------------------------------------

step         = CHUNK_DURATION - OVERLAP
chunk_starts = np.arange(0, duration - OVERLAP, step)
chunks       = [(round(cs, 2), round(min(cs + CHUNK_DURATION, duration), 2)) for cs in chunk_starts]

# --------------------------------------------------
# FLAG EVENTS (vectorized)
# --------------------------------------------------

frame_timestamps = np.array([f[0] for f in frames])

anomaly_mask = (
    (flow_var_arr      >= flow_var_threshold)      |
    (flow_collapse_arr >= flow_collapse_threshold) |
    (brightness_scores >= brightness_threshold)    |
    (scene_delta_arr   >= scene_delta_threshold)
)

chunk_flags = []
for cs, ce in chunks:
    in_chunk = (frame_timestamps >= cs) & (frame_timestamps <= ce)
    chunk_flags.append(bool(np.any(anomaly_mask & in_chunk)))

print(f"Event chunks: {sum(chunk_flags)} / {len(chunks)}")

# --------------------------------------------------
# FRAME SAMPLING
# --------------------------------------------------

def sample_frames(start, end, is_event):
    candidates = []

    for i, (ts, _, _, rgb) in enumerate(frames):
        if start <= ts <= end:
            importance = (
                flow_var_arr[i]
                + flow_collapse_arr[i]
                + brightness_scores[i]
                + scene_delta_arr[i]
            )
            candidates.append((i, importance, rgb))

    if not candidates:
        return []

    if not is_event:
        mid = candidates[len(candidates) // 2]
        return [Image.fromarray(mid[2])]

    candidates_sorted = sorted(candidates, key=lambda x: x[1], reverse=True)
    top_frames   = candidates_sorted[:2]
    start_frames = candidates[:1]
    end_frames   = candidates[-1:]

    unique = {}
    for i, score, rgb in (start_frames + top_frames + end_frames):
        unique[i] = rgb

    return [Image.fromarray(f) for f in list(unique.values())[:MAX_FRAMES_EVENT]]

# --------------------------------------------------
# GRID BUILDER
# --------------------------------------------------

def make_grid(pil_images, frame_width=640, frame_height=360):
    resized = [img.resize((frame_width, frame_height)) for img in pil_images]
    grid    = Image.new("RGB", (frame_width * len(resized), frame_height))
    for idx, img in enumerate(resized):
        grid.paste(img, (idx * frame_width, 0))
    return grid

# --------------------------------------------------
# PROMPTS
# --------------------------------------------------

SYSTEM_PROMPT = """
You are a dashcam footage analyst.

STRICT RULES:
- Base your description ONLY on what is directly visible in the provided frames.
- Do NOT carry forward objects or events from the previous scene unless they are still visible in the current frames.
- If the previous scene described something that is NOT visible in the current frames, do not mention it.
- Do NOT speculate or infer. If you cannot see it clearly, do not state it.

Left = driver's left (oncoming traffic side).
Right = driver's right (shoulder/curb side).

Look for:
- vehicles and their positions, especially any leaving the lane or road
- pedestrians appearing in or near the road and identify their occupation if visible, such as police officer, military personnel, pedestrian
- road infrastructure: cones, barriers, checkpoints, stop signs, roadblocks
- abnormal events: swerving, lane departure, wrong-way vehicles, sudden stops, emergency lights, smoke
"""

NORMAL_PROMPT = """
Previous scene context (for orientation only — do NOT repeat it):
{prev}

Current segment: {start}-{end}s.

Describe ONLY what is visible in the provided frame. One concise sentence.
"""

BATCH_NORMAL_PROMPT = """
Previous scene context (for orientation only — do NOT repeat it):
{prev}

You are shown two consecutive dashcam frames.
Frame 1 corresponds to segment {start}-{mid}s.
Frame 2 corresponds to segment {mid}-{end}s.

Write exactly TWO sentences, one per line:
Line 1: describe only what is visible in Frame 1.
Line 2: describe only what is visible in Frame 2.
"""

EVENT_PROMPT = """
Previous scene context (for orientation only — do NOT repeat it):
{prev}

Current segment: {start}-{end}s.

Something unusual may be occurring. Look carefully at ALL vehicles and people in the frame.

Pay specific attention to:
- any vehicle drifting, swerving, or leaving the paved road
- any vehicle crossing a lane line or moving onto the shoulder or grass
- any person appearing in or near the road and identify their occupation if visible
- any vehicle braking hard or stopping suddenly
- any infrastructure like checkpoints, barriers, or cones appearing

Do NOT mention anything from the previous scene that is not visible in these frames.
Write ONE concise sentence describing exactly what you see.
"""

# --------------------------------------------------
# CONTEXT BUILDER
# --------------------------------------------------

def build_context(text, start, end):
    short = text[:MEMORY_CHARS].strip()
    if len(text) > MEMORY_CHARS and " " in short:
        short = short.rsplit(" ", 1)[0]
    return f"{start}-{end}s: {short}"

# --------------------------------------------------
# MODEL INFERENCE — SINGLE
# --------------------------------------------------

def analyze_chunk(frame_imgs, start, end, is_event, prev):
    prompt = (EVENT_PROMPT if is_event else NORMAL_PROMPT).format(
        start=start, end=end, prev=prev
    )
    tokens = EVENT_TOKENS if is_event else NORMAL_TOKENS

    image_msgs = [{"type": "image", "image": f} for f in frame_imgs]

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": image_msgs + [{"type": "text", "text": prompt}]}
    ]

    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    image_inputs, _ = process_vision_info(messages)

    inputs = processor(
        text=[text],
        images=image_inputs,
        padding=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=tokens,
            temperature=0,
            do_sample=False
        )

    trimmed = output[:, inputs["input_ids"].shape[1]:]
    result  = processor.batch_decode(trimmed, skip_special_tokens=True)[0]

    del inputs, output
    gc.collect()
    torch.cuda.empty_cache()

    return result.strip()

# --------------------------------------------------
# RUN PIPELINE
# --------------------------------------------------

timeline     = []
prev_context = "start of video"

i = 0
while i < len(chunks):
    start, end = chunks[i]
    is_event   = chunk_flags[i]

    # --- Event: always solo ---
    if is_event:
        print(f"Chunk {i+1}/{len(chunks)} [{start}-{end}s] event=True")
        frame_imgs = sample_frames(start, end, True)
        if frame_imgs:
            text         = analyze_chunk(frame_imgs, start, end, True, prev_context)
            prev_context = build_context(text, start, end)
            timeline.append((start, end, True, text))
        i += 1

    # --- Normal: batch greedily up to NORMAL_BATCH_SIZE ---
    else:
        batch_indices = []
        j = i
        while j < len(chunks) and not chunk_flags[j] and len(batch_indices) < NORMAL_BATCH_SIZE:
            batch_indices.append(j)
            j += 1

        # single normal chunk — run solo
        if len(batch_indices) == 1:
            print(f"Chunk {i+1}/{len(chunks)} [{start}-{end}s] event=False")
            frame_imgs = sample_frames(start, end, False)
            if frame_imgs:
                text         = analyze_chunk(frame_imgs, start, end, False, prev_context)
                prev_context = build_context(text, start, end)
                timeline.append((start, end, False, text))
            i += 1
            continue

        # batch of normal chunks — grid inference
        batch_chunk_defs = [chunks[k] for k in batch_indices]
        batch_frame_list = [sample_frames(chunks[k][0], chunks[k][1], False) for k in batch_indices]

        valid = [(c, f[0]) for c, f in zip(batch_chunk_defs, batch_frame_list) if f]
        if not valid:
            i = j
            continue

        valid_chunks, valid_frames = zip(*valid)
        valid_chunks = list(valid_chunks)
        valid_frames = list(valid_frames)

        s0, e0 = valid_chunks[0]
        sN, eN = valid_chunks[-1]
        mid    = round((s0 + eN) / 2, 2)

        print(f"Chunks {i+1}-{i+len(valid_chunks)}/{len(chunks)} [{s0}-{eN}s] BATCH NORMAL")

        grid   = make_grid(valid_frames)
        prompt = BATCH_NORMAL_PROMPT.format(
            prev=prev_context,
            start=s0,
            mid=mid,
            end=eN,
        )

        image_msgs      = [{"type": "image", "image": grid}]
        messages        = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": image_msgs + [{"type": "text", "text": prompt}]}
        ]
        text_inp        = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, _ = process_vision_info(messages)
        inputs          = processor(text=[text_inp], images=image_inputs, padding=True, return_tensors="pt").to(model.device)

        with torch.inference_mode():
            output = model.generate(**inputs, max_new_tokens=NORMAL_TOKENS, temperature=0, do_sample=False)

        trimmed = output[:, inputs["input_ids"].shape[1]:]
        raw     = processor.batch_decode(trimmed, skip_special_tokens=True)[0].strip()

        del inputs, output
        gc.collect()
        torch.cuda.empty_cache()

        # parse two lines — one per chunk
        lines = [l.strip() for l in raw.splitlines() if l.strip()]
        while len(lines) < len(valid_chunks):
            lines.append(lines[-1] if lines else "Scene continues.")
        lines = lines[:len(valid_chunks)]

        for (s, e), line in zip(valid_chunks, lines):
            prev_context = build_context(line, s, e)
            timeline.append((s, e, False, line))

        i = j

# --------------------------------------------------
# PRINT TIMELINE
# --------------------------------------------------

print("\nTIMELINE\n")

for s, e, ev, t in timeline:
    tag = " [EVENT]" if ev else ""
    print(f"[{s}-{e}s]{tag} {t}")

print(f"\nTotal runtime: {time.time() - start_time:.1f}s")

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

Video duration: 69.53s  |  FPS: 30.0  |  Frames: 2086
Frames extracted: 694
Flow variance threshold:  1.2883
Flow collapse threshold:  0.0628
Brightness threshold:     0.0587
Scene delta threshold:    1.3417
Event chunks: 29 / 35
Chunk 1/35 [0.0-2.5s] event=True
Chunk 2/35 [2.0-4.5s] event=True
Chunk 3/35 [4.0-6.5s] event=True
Chunk 4/35 [6.0-8.5s] event=True
Chunk 5/35 [8.0-10.5s] event=True
Chunk 6/35 [10.0-12.5s] event=True
Chunk 7/35 [12.0-14.5s] event=True
Chunk 8/35 [14.0-16.5s] event=True
Chunk 9/35 [16.0-18.5s] event=True
Chunk 10/35 [18.0-20.5s] event=False
Chunk 11/35 [20.0-22.5s] event=True
Chunk 12/35 [22.0-24.5s] event=True
Chunk 13/35 [24.0-26.5s] event=True
Chunk 14/35 [26.0-28.5s] event=True
Chunk 15/35 [28.0-30.5s] event=True
Chunk 16/35 [30.0-32.5s] event=True
Chunk 17/35 [32.0-34.5s] event=True
Chunk 18/35 [34.0-36.5s] event=True
Chunk 19/35 [36.0-38.5s] event=True
Chunks 20-21/35 [38.0-42.5s] BATCH NORMAL
Chunk 22/35 [42.0-44.5s] event=True
Chunk 23/35 [44.0-46.5s] 

Trying batching again for normal consecutive frames, works well!

In [ ]:
# ===== Dashcam Analyzer v23 + Grid Batching for Normal Chunks =====

import torch
import cv2
import numpy as np
import gc
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText
from qwen_vl_utils import process_vision_info
import time

VIDEO_PATH = "/content/ambulance.mp4"
MODEL_ID   = "Qwen/Qwen3-VL-8B-Instruct"

# --------------------------------------------------
# Load model
# --------------------------------------------------

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

# --------------------------------------------------
# CONFIG
# --------------------------------------------------

start_time = time.time()
CHUNK_DURATION = 2.5
OVERLAP        = 0.5

FLOW_FPS          = 8
MAX_FRAMES_NORMAL = 1
MAX_FRAMES_EVENT  = 4

NORMAL_TOKENS     = 40
EVENT_TOKENS      = 120
MEMORY_CHARS      = 40
NORMAL_BATCH_SIZE = 2      # only change — batch consecutive normal chunks

FLOW_VAR_STD_MULT      = 1.0
BRIGHTNESS_STD_MULT    = 1.8
SCENE_DELTA_STD_MULT   = 1.0
FLOW_COLLAPSE_STD_MULT = 1.5

# --------------------------------------------------
# VIDEO INFO
# --------------------------------------------------

cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise RuntimeError(f"Could not open video: {VIDEO_PATH}")

native_fps   = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

if native_fps == 0:
    raise RuntimeError("Could not read FPS — check video file format/codec.")

duration = total_frames / native_fps

print(f"Video duration: {duration:.2f}s  |  FPS: {native_fps}  |  Frames: {total_frames}")

# --------------------------------------------------
# FRAME EXTRACTION
# --------------------------------------------------

frame_interval = max(1, int(native_fps / FLOW_FPS))

frames    = []
frame_idx = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    if frame_idx % frame_interval == 0:
        ts         = frame_idx / native_fps
        rgb        = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        gray       = cv2.cvtColor(cv2.resize(frame, (640, 360)), cv2.COLOR_BGR2GRAY)
        brightness = float(gray.mean())
        frames.append((ts, gray, brightness, rgb))

    frame_idx += 1

cap.release()
print(f"Frames extracted: {len(frames)}")

# --------------------------------------------------
# SIGNAL 1+2: FLOW VARIANCE + FORWARD FLOW COLLAPSE
# --------------------------------------------------

flow_variance_scores = [0.0]
flow_collapse_scores = [0.0]
prev_forward_flow    = None

for i in range(1, len(frames)):
    prev_gray = frames[i-1][1]
    curr_gray = frames[i][1]

    flow = cv2.calcOpticalFlowFarneback(
        prev_gray, curr_gray, None,
        0.5, 3, 15, 3, 5, 1.2, 0
    )

    mag, _ = cv2.cartToPolar(flow[..., 0], flow[..., 1])

    flow_variance = float(np.std(mag))
    flow_variance_scores.append(flow_variance)

    forward_flow = float(np.mean(np.abs(flow[..., 1])))
    collapse     = max(0.0, prev_forward_flow - forward_flow) if prev_forward_flow is not None else 0.0
    flow_collapse_scores.append(collapse)
    prev_forward_flow = forward_flow

flow_var_arr      = np.array(flow_variance_scores)
flow_collapse_arr = np.array(flow_collapse_scores)

flow_var_threshold      = flow_var_arr.mean()      + FLOW_VAR_STD_MULT      * flow_var_arr.std()
flow_collapse_threshold = flow_collapse_arr.mean() + FLOW_COLLAPSE_STD_MULT * flow_collapse_arr.std()

print(f"Flow variance threshold:  {flow_var_threshold:.4f}")
print(f"Flow collapse threshold:  {flow_collapse_threshold:.4f}")

# --------------------------------------------------
# SIGNAL 3: BRIGHTNESS SECOND DERIVATIVE
# --------------------------------------------------

brightness_arr    = np.array([f[2] for f in frames])
d1                = np.abs(np.diff(brightness_arr))
d2                = np.abs(np.diff(d1))
brightness_scores = np.concatenate([[0, 0], d2])

brightness_threshold = brightness_scores.mean() + BRIGHTNESS_STD_MULT * brightness_scores.std()

print(f"Brightness threshold:     {brightness_threshold:.4f}")

# --------------------------------------------------
# SIGNAL 4: SCENE DELTA
# --------------------------------------------------

ROLLING_WINDOW = 8

scene_delta_scores = []

for i in range(len(frames)):
    curr_gray    = frames[i][1].astype(np.float32)
    window_start = max(0, i - ROLLING_WINDOW)
    window_grays = np.array([frames[j][1].astype(np.float32) for j in range(window_start, i)])

    if len(window_grays) == 0:
        scene_delta_scores.append(0.0)
    else:
        rolling_mean = window_grays.mean(axis=0)
        delta        = float(np.mean(np.abs(curr_gray - rolling_mean)))
        scene_delta_scores.append(delta)

scene_delta_arr       = np.array(scene_delta_scores)
scene_delta_threshold = scene_delta_arr.mean() + SCENE_DELTA_STD_MULT * scene_delta_arr.std()

print(f"Scene delta threshold:    {scene_delta_threshold:.4f}")

# --------------------------------------------------
# BUILD CHUNKS
# --------------------------------------------------

step         = CHUNK_DURATION - OVERLAP
chunk_starts = np.arange(0, duration - OVERLAP, step)
chunks       = [(round(cs, 2), round(min(cs + CHUNK_DURATION, duration), 2)) for cs in chunk_starts]

# --------------------------------------------------
# FLAG EVENTS (vectorized)
# --------------------------------------------------

frame_timestamps = np.array([f[0] for f in frames])

anomaly_mask = (
    (flow_var_arr      >= flow_var_threshold)      |
    (flow_collapse_arr >= flow_collapse_threshold) |
    (brightness_scores >= brightness_threshold)    |
    (scene_delta_arr   >= scene_delta_threshold)
)

chunk_flags = []
for cs, ce in chunks:
    in_chunk = (frame_timestamps >= cs) & (frame_timestamps <= ce)
    chunk_flags.append(bool(np.any(anomaly_mask & in_chunk)))

print(f"Event chunks: {sum(chunk_flags)} / {len(chunks)}")

# --------------------------------------------------
# FRAME SAMPLING
# --------------------------------------------------

def sample_frames(start, end, is_event):
    candidates = []

    for i, (ts, _, _, rgb) in enumerate(frames):
        if start <= ts <= end:
            importance = (
                flow_var_arr[i]
                + flow_collapse_arr[i]
                + brightness_scores[i]
                + scene_delta_arr[i]
            )
            candidates.append((i, importance, rgb))

    if not candidates:
        return []

    if not is_event:
        mid = candidates[len(candidates) // 2]
        return [Image.fromarray(mid[2])]

    candidates_sorted = sorted(candidates, key=lambda x: x[1], reverse=True)
    top_frames   = candidates_sorted[:2]
    start_frames = candidates[:1]
    end_frames   = candidates[-1:]

    unique = {}
    for i, score, rgb in (start_frames + top_frames + end_frames):
        unique[i] = rgb

    return [Image.fromarray(f) for f in list(unique.values())[:MAX_FRAMES_EVENT]]

# --------------------------------------------------
# GRID BUILDER
# --------------------------------------------------

def make_grid(pil_images, frame_width=640, frame_height=360):
    resized = [img.resize((frame_width, frame_height)) for img in pil_images]
    grid    = Image.new("RGB", (frame_width * len(resized), frame_height))
    for idx, img in enumerate(resized):
        grid.paste(img, (idx * frame_width, 0))
    return grid

# --------------------------------------------------
# PROMPTS
# --------------------------------------------------

SYSTEM_PROMPT = """
You are a dashcam footage analyst.

STRICT RULES:
- Base your description ONLY on what is directly visible in the provided frames.
- Do NOT carry forward objects or events from the previous scene unless they are still visible in the current frames.
- If the previous scene described something that is NOT visible in the current frames, do not mention it.
- Do NOT speculate or infer. If you cannot see it clearly, do not state it.

Left = driver's left (oncoming traffic side).
Right = driver's right (shoulder/curb side).

Look for:
- vehicles and their positions, especially any leaving the lane or road
- pedestrians appearing in or near the road and idenfity their occupation if visible, such as police officer, military personnel, pedestrian
- road infrastructure: cones, barriers, checkpoints, stop signs, roadblocks
- abnormal events: swerving, lane departure, wrong-way vehicles, sudden stops, emergency lights, smoke
"""

NORMAL_PROMPT = """
Previous scene context (for orientation only — do NOT repeat it):
{prev}

Current segment: {start}-{end}s.

Describe ONLY what is visible in the provided frame. One concise sentence.
"""

BATCH_NORMAL_PROMPT = """
Previous scene context (for orientation only — do NOT repeat it):
{prev}

Current segments: {start}-{end}s.

You are shown a side-by-side grid of {n} consecutive dashcam frames (left = earlier, right = later).

Describe ONLY what is visible across both frames. One concise sentence.
"""

EVENT_PROMPT = """
Previous scene context (for orientation only — do NOT repeat it):
{prev}

Current segment: {start}-{end}s.

Something unusual may be occurring. Look carefully at ALL vehicles and people in the frame.

Pay specific attention to:
- any vehicle drifting, swerving, or leaving the paved road
- any vehicle crossing a lane line or moving onto the shoulder or grass
- any person appearing in or near the road and identify their occupation if visible
- any vehicle braking hard or stopping suddenly
- any infrastructure like checkpoints, barriers, or cones appearing

Do NOT mention anything from the previous scene that is not visible in these frames.
Write ONE concise sentence describing exactly what you see.
"""

# --------------------------------------------------
# CONTEXT BUILDER
# --------------------------------------------------

def build_context(text, start, end):
    short = text[:MEMORY_CHARS].strip()
    if len(text) > MEMORY_CHARS and " " in short:
        short = short.rsplit(" ", 1)[0]
    return f"{start}-{end}s: {short}"

# --------------------------------------------------
# MODEL INFERENCE — SINGLE
# --------------------------------------------------

def analyze_chunk(frame_imgs, start, end, is_event, prev):
    prompt = (EVENT_PROMPT if is_event else NORMAL_PROMPT).format(
        start=start, end=end, prev=prev
    )
    tokens = EVENT_TOKENS if is_event else NORMAL_TOKENS

    image_msgs = [{"type": "image", "image": f} for f in frame_imgs]

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": image_msgs + [{"type": "text", "text": prompt}]}
    ]

    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    image_inputs, _ = process_vision_info(messages)

    inputs = processor(
        text=[text],
        images=image_inputs,
        padding=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=tokens,
            temperature=0,
            do_sample=False
        )

    trimmed = output[:, inputs["input_ids"].shape[1]:]
    result  = processor.batch_decode(trimmed, skip_special_tokens=True)[0]

    del inputs, output
    gc.collect()
    torch.cuda.empty_cache()

    return result.strip()

# --------------------------------------------------
# RUN PIPELINE
# --------------------------------------------------

timeline     = []
prev_context = "start of video"

i = 0
while i < len(chunks):
    start, end = chunks[i]
    is_event   = chunk_flags[i]

    # --- Event: always solo ---
    if is_event:
        print(f"Chunk {i+1}/{len(chunks)} [{start}-{end}s] event=True")
        frame_imgs = sample_frames(start, end, True)
        if frame_imgs:
            text         = analyze_chunk(frame_imgs, start, end, True, prev_context)
            prev_context = build_context(text, start, end)
            timeline.append((start, end, True, text))
        i += 1

    # --- Normal: batch greedily up to NORMAL_BATCH_SIZE ---
    else:
        batch_indices = []
        j = i
        while j < len(chunks) and not chunk_flags[j] and len(batch_indices) < NORMAL_BATCH_SIZE:
            batch_indices.append(j)
            j += 1

        # single normal chunk — run solo
        if len(batch_indices) == 1:
            print(f"Chunk {i+1}/{len(chunks)} [{start}-{end}s] event=False")
            frame_imgs = sample_frames(start, end, False)
            if frame_imgs:
                text         = analyze_chunk(frame_imgs, start, end, False, prev_context)
                prev_context = build_context(text, start, end)
                timeline.append((start, end, False, text))
            i += 1
            continue

        # batch of normal chunks — grid inference
        batch_chunk_defs = [chunks[k] for k in batch_indices]
        batch_frame_list = [sample_frames(chunks[k][0], chunks[k][1], False) for k in batch_indices]

        valid = [(c, f[0]) for c, f in zip(batch_chunk_defs, batch_frame_list) if f]
        if not valid:
            i = j
            continue

        valid_chunks, valid_frames = zip(*valid)
        valid_chunks = list(valid_chunks)
        valid_frames = list(valid_frames)

        s0, e0 = valid_chunks[0]
        sN, eN = valid_chunks[-1]
        print(f"Chunks {i+1}-{i+len(valid_chunks)}/{len(chunks)} [{s0}-{eN}s] BATCH NORMAL")

        grid   = make_grid(valid_frames)
        prompt = BATCH_NORMAL_PROMPT.format(
            prev=prev_context,
            n=len(valid_chunks),
            start=s0,
            end=eN,
        )

        image_msgs      = [{"type": "image", "image": grid}]
        messages        = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": image_msgs + [{"type": "text", "text": prompt}]}
        ]
        text_inp        = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, _ = process_vision_info(messages)
        inputs          = processor(text=[text_inp], images=image_inputs, padding=True, return_tensors="pt").to(model.device)

        with torch.inference_mode():
            output = model.generate(**inputs, max_new_tokens=NORMAL_TOKENS, temperature=0, do_sample=False)

        trimmed = output[:, inputs["input_ids"].shape[1]:]
        text    = processor.batch_decode(trimmed, skip_special_tokens=True)[0].strip()

        del inputs, output
        gc.collect()
        torch.cuda.empty_cache()

        prev_context = build_context(text, sN, eN)
        for s, e in valid_chunks:
            timeline.append((s, e, False, text))

        i = j

# --------------------------------------------------
# PRINT TIMELINE
# --------------------------------------------------

print("\nTIMELINE\n")

for s, e, ev, t in timeline:
    tag = " [EVENT]" if ev else ""
    print(f"[{s}-{e}s]{tag} {t}")

print(f"\nTotal runtime: {time.time() - start_time:.1f}s")

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

Video duration: 27.16s  |  FPS: 29.97002997002997  |  Frames: 814
Frames extracted: 271
Flow variance threshold:  2.6704
Flow collapse threshold:  0.1038
Brightness threshold:     0.2076
Scene delta threshold:    3.1169
Event chunks: 7 / 14
Chunk 1/14 [0.0-2.5s] event=False
Chunk 2/14 [2.0-4.5s] event=True
Chunk 3/14 [4.0-6.5s] event=True
Chunks 4-5/14 [6.0-10.5s] BATCH NORMAL
Chunks 6-7/14 [10.0-14.5s] BATCH NORMAL
Chunks 8-9/14 [14.0-18.5s] BATCH NORMAL
Chunk 10/14 [18.0-20.5s] event=True
Chunk 11/14 [20.0-22.5s] event=True
Chunk 12/14 [22.0-24.5s] event=True
Chunk 13/14 [24.0-26.5s] event=True
Chunk 14/14 [26.0-27.16s] event=True

TIMELINE

[0.0-2.5s] A wide intersection with multiple lanes is visible, featuring a white van and a black car in the distance, a black car turning left from the top-left, and a cyclist on a path in the background
[2.0-4.5s] [EVENT] A dark gray SUV is turning right across the intersection, crossing from the left side of the frame onto the right side.
[4.0-

Improved + Getting more efficient -> This is pretty accurate and fast so far

In [ ]:
# ===== Dashcam Analyzer v23 — Raw Video, No Annotations =====

import torch
import cv2
import numpy as np
import gc
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText
from qwen_vl_utils import process_vision_info
import time

VIDEO_PATH = "/content/military.mp4"
MODEL_ID   = "Qwen/Qwen3-VL-8B-Instruct"

# --------------------------------------------------
# Load model
# --------------------------------------------------

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

# --------------------------------------------------
# CONFIG
# --------------------------------------------------

start_time = time.time()
CHUNK_DURATION = 2.5
OVERLAP        = 0.5

FLOW_FPS          = 8
MAX_FRAMES_NORMAL = 1
MAX_FRAMES_EVENT  = 4

NORMAL_TOKENS = 40
EVENT_TOKENS  = 120
MEMORY_CHARS  = 40

FLOW_VAR_STD_MULT      = 1.0
BRIGHTNESS_STD_MULT    = 1.8
SCENE_DELTA_STD_MULT   = 1.0
FLOW_COLLAPSE_STD_MULT = 1.5

# --------------------------------------------------
# VIDEO INFO
# --------------------------------------------------

cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise RuntimeError(f"Could not open video: {VIDEO_PATH}")

native_fps   = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

if native_fps == 0:
    raise RuntimeError("Could not read FPS — check video file format/codec.")

duration = total_frames / native_fps

print(f"Video duration: {duration:.2f}s  |  FPS: {native_fps}  |  Frames: {total_frames}")

# --------------------------------------------------
# FRAME EXTRACTION
# --------------------------------------------------

frame_interval = max(1, int(native_fps / FLOW_FPS))

frames    = []
frame_idx = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    if frame_idx % frame_interval == 0:
        ts         = frame_idx / native_fps
        rgb        = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        gray       = cv2.cvtColor(cv2.resize(frame, (640, 360)), cv2.COLOR_BGR2GRAY)
        brightness = float(gray.mean())
        frames.append((ts, gray, brightness, rgb))

    frame_idx += 1

cap.release()
print(f"Frames extracted: {len(frames)}")

# --------------------------------------------------
# SIGNAL 1+2: FLOW VARIANCE + FORWARD FLOW COLLAPSE
# --------------------------------------------------

flow_variance_scores = [0.0]
flow_collapse_scores = [0.0]
prev_forward_flow    = None

for i in range(1, len(frames)):
    prev_gray = frames[i-1][1]
    curr_gray = frames[i][1]

    flow = cv2.calcOpticalFlowFarneback(
        prev_gray, curr_gray, None,
        0.5, 3, 15, 3, 5, 1.2, 0
    )

    mag, _ = cv2.cartToPolar(flow[..., 0], flow[..., 1])

    flow_variance = float(np.std(mag))
    flow_variance_scores.append(flow_variance)

    forward_flow = float(np.mean(np.abs(flow[..., 1])))
    collapse     = max(0.0, prev_forward_flow - forward_flow) if prev_forward_flow is not None else 0.0
    flow_collapse_scores.append(collapse)
    prev_forward_flow = forward_flow

flow_var_arr      = np.array(flow_variance_scores)
flow_collapse_arr = np.array(flow_collapse_scores)

flow_var_threshold      = flow_var_arr.mean()      + FLOW_VAR_STD_MULT      * flow_var_arr.std()
flow_collapse_threshold = flow_collapse_arr.mean() + FLOW_COLLAPSE_STD_MULT * flow_collapse_arr.std()

print(f"Flow variance threshold:  {flow_var_threshold:.4f}")
print(f"Flow collapse threshold:  {flow_collapse_threshold:.4f}")

# --------------------------------------------------
# SIGNAL 3: BRIGHTNESS SECOND DERIVATIVE
# --------------------------------------------------

brightness_arr    = np.array([f[2] for f in frames])
d1                = np.abs(np.diff(brightness_arr))
d2                = np.abs(np.diff(d1))
brightness_scores = np.concatenate([[0, 0], d2])

brightness_threshold = brightness_scores.mean() + BRIGHTNESS_STD_MULT * brightness_scores.std()

print(f"Brightness threshold:     {brightness_threshold:.4f}")

# --------------------------------------------------
# SIGNAL 4: SCENE DELTA
# --------------------------------------------------

ROLLING_WINDOW = 8

scene_delta_scores = []

for i in range(len(frames)):
    curr_gray    = frames[i][1].astype(np.float32)
    window_start = max(0, i - ROLLING_WINDOW)
    window_grays = np.array([frames[j][1].astype(np.float32) for j in range(window_start, i)])

    if len(window_grays) == 0:
        scene_delta_scores.append(0.0)
    else:
        rolling_mean = window_grays.mean(axis=0)
        delta        = float(np.mean(np.abs(curr_gray - rolling_mean)))
        scene_delta_scores.append(delta)

scene_delta_arr       = np.array(scene_delta_scores)
scene_delta_threshold = scene_delta_arr.mean() + SCENE_DELTA_STD_MULT * scene_delta_arr.std()

print(f"Scene delta threshold:    {scene_delta_threshold:.4f}")

# --------------------------------------------------
# BUILD CHUNKS
# --------------------------------------------------

step         = CHUNK_DURATION - OVERLAP
chunk_starts = np.arange(0, duration - OVERLAP, step)
chunks       = [(round(cs, 2), round(min(cs + CHUNK_DURATION, duration), 2)) for cs in chunk_starts]

# --------------------------------------------------
# FLAG EVENTS (vectorized)
# --------------------------------------------------

frame_timestamps = np.array([f[0] for f in frames])

anomaly_mask = (
    (flow_var_arr      >= flow_var_threshold)      |
    (flow_collapse_arr >= flow_collapse_threshold) |
    (brightness_scores >= brightness_threshold)    |
    (scene_delta_arr   >= scene_delta_threshold)
)

chunk_flags = []
for cs, ce in chunks:
    in_chunk = (frame_timestamps >= cs) & (frame_timestamps <= ce)
    chunk_flags.append(bool(np.any(anomaly_mask & in_chunk)))

print(f"Event chunks: {sum(chunk_flags)} / {len(chunks)}")

# --------------------------------------------------
# FRAME SAMPLING
# --------------------------------------------------

def sample_frames(start, end, is_event):
    candidates = []

    for i, (ts, _, _, rgb) in enumerate(frames):
        if start <= ts <= end:
            importance = (
                flow_var_arr[i]
                + flow_collapse_arr[i]
                + brightness_scores[i]
                + scene_delta_arr[i]
            )
            candidates.append((i, importance, rgb))

    if not candidates:
        return []

    if not is_event:
        mid = candidates[len(candidates) // 2]
        return [Image.fromarray(mid[2])]

    candidates_sorted = sorted(candidates, key=lambda x: x[1], reverse=True)
    top_frames   = candidates_sorted[:2]
    start_frames = candidates[:1]
    end_frames   = candidates[-1:]

    unique = {}
    for i, score, rgb in (start_frames + top_frames + end_frames):
        unique[i] = rgb

    return [Image.fromarray(f) for f in list(unique.values())[:MAX_FRAMES_EVENT]]

# --------------------------------------------------
# PROMPTS
# --------------------------------------------------

SYSTEM_PROMPT = """
You are a dashcam footage analyst.

STRICT RULES:
- Base your description ONLY on what is directly visible in the provided frames.
- Do NOT carry forward objects or events from the previous scene unless they are still visible in the current frames.
- If the previous scene described something that is NOT visible in the current frames, do not mention it.
- Do NOT speculate or infer. If you cannot see it clearly, do not state it.

Left = driver's left (oncoming traffic side).
Right = driver's right (shoulder/curb side).

Look for:
- vehicles and their positions, especially any leaving the lane or road
- pedestrians appearing in or near the road and idenfity their occupation if visible, such as police officer, military personnel, pedestrian
- road infrastructure: cones, barriers, checkpoints, stop signs, roadblocks
- abnormal events: swerving, lane departure, wrong-way vehicles, sudden stops, emergency lights, smoke
"""

NORMAL_PROMPT = """
Previous scene context (for orientation only — do NOT repeat it):
{prev}

Current segment: {start}-{end}s.

Describe ONLY what is visible in the provided frame. One concise sentence.
"""

EVENT_PROMPT = """
Previous scene context (for orientation only — do NOT repeat it):
{prev}

Current segment: {start}-{end}s.

Something unusual may be occurring. Look carefully at ALL vehicles and people in the frame.

Pay specific attention to:
- any vehicle drifting, swerving, or leaving the paved road
- any vehicle crossing a lane line or moving onto the shoulder or grass
- any person appearing in or near the road and identify their occupation if visible
- any vehicle braking hard or stopping suddenly
- any infrastructure like checkpoints, barriers, or cones appearing

Do NOT mention anything from the previous scene that is not visible in these frames.
Write ONE concise sentence describing exactly what you see.
"""

# --------------------------------------------------
# CONTEXT BUILDER
# --------------------------------------------------

def build_context(text, start, end):
    short = text[:MEMORY_CHARS].strip()
    if len(text) > MEMORY_CHARS and " " in short:
        short = short.rsplit(" ", 1)[0]
    return f"{start}-{end}s: {short}"

# --------------------------------------------------
# MODEL INFERENCE
# --------------------------------------------------

def analyze_chunk(frame_imgs, start, end, is_event, prev):
    prompt = (EVENT_PROMPT if is_event else NORMAL_PROMPT).format(
        start=start, end=end, prev=prev
    )
    tokens = EVENT_TOKENS if is_event else NORMAL_TOKENS

    image_msgs = [{"type": "image", "image": f} for f in frame_imgs]

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": image_msgs + [{"type": "text", "text": prompt}]}
    ]

    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    image_inputs, _ = process_vision_info(messages)

    inputs = processor(
        text=[text],
        images=image_inputs,
        padding=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=tokens,
            temperature=0,
            do_sample=False
        )

    trimmed = output[:, inputs["input_ids"].shape[1]:]
    result  = processor.batch_decode(trimmed, skip_special_tokens=True)[0]

    del inputs, output
    gc.collect()
    torch.cuda.empty_cache()

    return result.strip()

# --------------------------------------------------
# RUN PIPELINE
# --------------------------------------------------

timeline     = []
prev_context = "start of video"

for i, (start, end) in enumerate(chunks):
    is_event = chunk_flags[i]
    print(f"Chunk {i+1}/{len(chunks)} [{start}-{end}s] event={is_event}")

    frame_imgs = sample_frames(start, end, is_event)
    if not frame_imgs:
        continue

    text         = analyze_chunk(frame_imgs, start, end, is_event, prev_context)
    prev_context = build_context(text, start, end)
    timeline.append((start, end, is_event, text))

# --------------------------------------------------
# PRINT TIMELINE
# --------------------------------------------------

print("\nTIMELINE\n")

for s, e, ev, t in timeline:
    tag = " [EVENT]" if ev else ""
    print(f"[{s}-{e}s]{tag} {t}")

print(f"\nTotal runtime: {time.time() - start_time:.1f}s")

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

Video duration: 97.00s  |  FPS: 30.0  |  Frames: 2910
Frames extracted: 970
Flow variance threshold:  4.0552
Flow collapse threshold:  0.2799
Brightness threshold:     1.1252
Scene delta threshold:    8.1778
Event chunks: 27 / 49
Chunk 1/49 [0.0-2.5s] event=False
Chunk 2/49 [2.0-4.5s] event=False
Chunk 3/49 [4.0-6.5s] event=True
Chunk 4/49 [6.0-8.5s] event=False
Chunk 5/49 [8.0-10.5s] event=False
Chunk 6/49 [10.0-12.5s] event=False
Chunk 7/49 [12.0-14.5s] event=False
Chunk 8/49 [14.0-16.5s] event=False
Chunk 9/49 [16.0-18.5s] event=False
Chunk 10/49 [18.0-20.5s] event=False
Chunk 11/49 [20.0-22.5s] event=False
Chunk 12/49 [22.0-24.5s] event=False
Chunk 13/49 [24.0-26.5s] event=False
Chunk 14/49 [26.0-28.5s] event=False
Chunk 15/49 [28.0-30.5s] event=True
Chunk 16/49 [30.0-32.5s] event=False
Chunk 17/49 [32.0-34.5s] event=False
Chunk 18/49 [34.0-36.5s] event=True
Chunk 19/49 [36.0-38.5s] event=True
Chunk 20/49 [38.0-40.5s] event=True
Chunk 21/49 [40.0-42.5s] event=True
Chunk 22/49 [42.0

In [ ]:
# ===== Dashcam Analyzer v18 — Flow + Brightness + Trajectory Detection =====

import torch
import cv2
import numpy as np
import gc
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText
from qwen_vl_utils import process_vision_info
import time

VIDEO_PATH = "/content/dashcam.mp4"
MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"

# --------------------------------------------------
# Load model
# --------------------------------------------------

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

# --------------------------------------------------
# CONFIG
# --------------------------------------------------

start_time = time.time()
CHUNK_DURATION = 2.5   # slightly longer window
OVERLAP = 0.5     # was 1.0, keeps decent boundary coverage

FLOW_FPS = 8  # was 8
MAX_FRAMES = 5          # was 8
MAX_FRAMES_EVENT = 8    # events keep full frames

NORMAL_TOKENS = 60
EVENT_TOKENS = 150
MEMORY_CHARS = 60

FLOW_STD_MULT = 1.8
BRIGHTNESS_STD_MULT = 1.8
TRAJ_STD_MULT = 1.2

# --------------------------------------------------
# VIDEO INFO
# --------------------------------------------------

cap = cv2.VideoCapture(VIDEO_PATH)

native_fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
duration = total_frames / native_fps

print("Video duration:", duration)

# --------------------------------------------------
# FRAME EXTRACTION
# --------------------------------------------------

frame_interval = max(1, int(native_fps / FLOW_FPS))

frames = []

frame_idx = 0

while True:

    ret, frame = cap.read()

    if not ret:
        break

    if frame_idx % frame_interval == 0:

        ts = frame_idx / native_fps

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        gray = cv2.cvtColor(cv2.resize(frame,(640,360)), cv2.COLOR_BGR2GRAY)

        brightness = float(gray.mean())

        frames.append((ts, gray, brightness, rgb))

    frame_idx += 1

cap.release()

print("Frames extracted:", len(frames))

# --------------------------------------------------
# OPTICAL FLOW + TRAJECTORY SCORES
# --------------------------------------------------

flow_scores = [0.0]
trajectory_scores = [0.0]

for i in range(1, len(frames)):

    prev = frames[i-1][1]
    curr = frames[i][1]

    flow = cv2.calcOpticalFlowFarneback(
        prev, curr, None,
        0.5, 3, 15,
        3, 5, 1.2, 0
    )

    mag, ang = cv2.cartToPolar(flow[...,0], flow[...,1])

    median = np.median(mag)

    flow_anomaly = np.mean(np.maximum(mag - median, 0))

    sideways = np.mean(np.abs(flow[...,0]))
    forward = np.mean(np.abs(flow[...,1]))

    trajectory_score = sideways / (forward + 1e-5)

    flow_scores.append(float(flow_anomaly))
    trajectory_scores.append(float(trajectory_score))

# thresholds

flow_threshold = np.mean(flow_scores) + FLOW_STD_MULT * np.std(flow_scores)

traj_threshold = np.mean(trajectory_scores) + TRAJ_STD_MULT * np.std(trajectory_scores)

print("Flow threshold:", flow_threshold)
print("Trajectory threshold:", traj_threshold)

# --------------------------------------------------
# BRIGHTNESS SCORES
# --------------------------------------------------

brightness = np.array([f[2] for f in frames])

d1 = np.abs(np.diff(brightness))
d2 = np.abs(np.diff(d1))

brightness_scores = np.concatenate([[0,0], d2])

brightness_threshold = np.mean(brightness_scores) + BRIGHTNESS_STD_MULT * np.std(brightness_scores)

print("Brightness threshold:", brightness_threshold)

# --------------------------------------------------
# BUILD CHUNKS
# --------------------------------------------------

step = CHUNK_DURATION - OVERLAP

chunk_starts = np.arange(0, duration - OVERLAP, step)

chunks = []

for cs in chunk_starts:

    ce = min(cs + CHUNK_DURATION, duration)

    chunks.append((round(cs,2), round(ce,2)))

# --------------------------------------------------
# FLAG EVENTS
# --------------------------------------------------

chunk_flags = []

for cs, ce in chunks:

    flagged = False

    for i,(ts,_,_,_) in enumerate(frames):

        if cs <= ts <= ce:

            if (
                flow_scores[i] >= flow_threshold
                or brightness_scores[i] >= brightness_threshold
                or trajectory_scores[i] >= traj_threshold
            ):

                flagged = True

    chunk_flags.append(flagged)

# --------------------------------------------------
# FRAME SAMPLING
# --------------------------------------------------

def sample_frames(start, end, max_frames=MAX_FRAMES):

    candidates = []

    for i,(ts,_,_,rgb) in enumerate(frames):

        if start <= ts <= end:

            importance = (
                flow_scores[i]
                + brightness_scores[i]
                + trajectory_scores[i]
            )

            candidates.append((i, importance, rgb))

    if not candidates:
        return []

    candidates_sorted = sorted(candidates, key=lambda x: x[1], reverse=True)

    motion_frames = candidates_sorted[:4]
    start_frames = candidates[:2]
    end_frames = candidates[-2:]

    selected = start_frames + motion_frames + end_frames

    unique = {}
    for i,score,frame in selected:
        unique[i] = frame

    selected_frames = list(unique.values())[:max_frames]

    return [Image.fromarray(f) for f in selected_frames]

# --------------------------------------------------
# PROMPTS
# --------------------------------------------------

SYSTEM_PROMPT = """
You are analyzing dashcam footage sequentially.

Describe exactly what is visible.
Do not speculate.

Left and right are from the driver's perspective.
Left = driver's left (usually oncoming traffic side).
Right = driver's right (usually the shoulder/curb side).

Focus on:
- vehicles and where they are in the scene
- pedestrians and their occupation or role if visible
- traffic situation
- road infrastructure: traffic cones, barriers, checkpoints, stop signs, sandbags, roadblocks
- abnormal events (such as swerving, smoke, off-road vehicle, emergency lights)
"""

NORMAL_PROMPT = """
Previous scene:
{prev}

Segment {start}-{end}s.

Write ONE concise sentence describing the scene including any road infrastructure visible.
"""

EVENT_PROMPT = """
Previous scene:
{prev}

Segment {start}-{end}s.

Write ONE concise sentence describing the scene and any abnormal vehicle or pedestrian behavior.
Use the previous scene provided for context.

"""

# --------------------------------------------------
# MODEL INFERENCE
# --------------------------------------------------

def analyze_chunk(frames, start, end, is_event, prev):

    if is_event:
        prompt = EVENT_PROMPT.format(start=start, end=end, prev=prev)
        tokens = EVENT_TOKENS
        max_f = MAX_FRAMES_EVENT
    else:
        prompt = NORMAL_PROMPT.format(start=start, end=end, prev=prev)
        tokens = NORMAL_TOKENS
        max_f = MAX_FRAMES

    image_msgs = [{"type":"image","image":f} for f in frames]

    messages = [
        {"role":"system","content":SYSTEM_PROMPT},
        {"role":"user","content":image_msgs + [{"type":"text","text":prompt}]}
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    image_inputs,_ = process_vision_info(messages)

    inputs = processor(
        text=[text],
        images=image_inputs,
        padding=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():

        output = model.generate(
            **inputs,
            max_new_tokens=tokens,
            temperature=0.1
        )

    trimmed = output[:,inputs["input_ids"].shape[1]:]

    result = processor.batch_decode(trimmed,skip_special_tokens=True)[0]

    del inputs,output
    gc.collect()
    torch.cuda.empty_cache()

    return result.strip()

# --------------------------------------------------
# RUN PIPELINE
# --------------------------------------------------

timeline = []

prev_summary = "start of video"

for i,(start,end) in enumerate(chunks):

    print("Chunk",i+1,"/",len(chunks),"event:",chunk_flags[i])

    is_event = chunk_flags[i]
    max_f = MAX_FRAMES_EVENT if is_event else MAX_FRAMES
    frames_sample = sample_frames(start, end, max_frames=max_f)

    if not frames_sample:
        continue

    text = analyze_chunk(frames_sample, start, end, is_event, prev_summary)

    prev_summary = text[:MEMORY_CHARS]

    timeline.append((start, end, text))

# --------------------------------------------------
# PRINT TIMELINE
# --------------------------------------------------

print("\nTIMELINE\n")

for s,e,t in timeline:
    print(f"[{s}-{e}s] {t}")

print("Total runtime:", time.time() - start_time, "seconds")

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

Video duration: 31.66496666666667
Frames extracted: 316
Flow threshold: 1.4276340981626723
Trajectory threshold: 3.5641157396201764
Brightness threshold: 4.50034116600185
Chunk 1 / 16 event: False
Chunk 2 / 16 event: True
Chunk 3 / 16 event: True
Chunk 4 / 16 event: True
Chunk 5 / 16 event: True
Chunk 6 / 16 event: False
Chunk 7 / 16 event: True
Chunk 8 / 16 event: True
Chunk 9 / 16 event: True
Chunk 10 / 16 event: True
Chunk 11 / 16 event: False
Chunk 12 / 16 event: True
Chunk 13 / 16 event: True
Chunk 14 / 16 event: True
Chunk 15 / 16 event: True
Chunk 16 / 16 event: False

TIMELINE

[0.0-2.5s] At night, a white SUV with flashing red lights is stopped on the left side of a two-lane road, with a solid white line marking the edge of the lane and vegetation on the right shoulder.
[2.0-4.5s] A white SUV with flashing red lights is stopped on the right shoulder of a dark road, and a large truck passes closely on the left, obscuring the view of the SUV.
[4.0-6.5s] A large semi-truck passes